In [5]:
# ============================================================
# 11_vol_forecast_denominator_v0_1
#
# Goal:
#   Build the target/feature panel needed for volatility
#   forecasting.
#
# This clean restart only does:
#   1. Load existing VRP panel
#   2. Recover/standardize tenor
#   3. Build unique SPX return table
#   4. Prepare for forward-RV targets and trailing-RV features
#
# No model fitting yet.
# No new trade signals yet.
# No pricing.
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

# ----------------------------
# Paths
# ----------------------------

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

VRP_PANEL_PARQUET = PROCESSED_DATA_DIR / "vrp_panel_v0_1.parquet"

VIX_TERM_STRUCTURE_PARQUET = (
    PROCESSED_DATA_DIR
    / "vix_term_structure_history_v0_7_1_repaired_total_variance.parquet"
)

VOL_FORECAST_FEATURE_TARGET_PARQUET = (
    PROCESSED_DATA_DIR / "vol_forecast_feature_target_panel_v0_1.parquet"
)

VOL_FORECAST_FEATURE_TARGET_AUDIT_CSV = (
    AUDIT_DIR / "vol_forecast_feature_target_audit_v0_1.csv"
)

TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

FEATURE_WINDOWS = [
    5, 9, 12, 15, 18, 21, 24, 27, 30, 33, 42, 63, 126, 252
]

EPSILON_VARIANCE = 1e-10

print("Project root:", PROJECT_ROOT)
print("VRP panel exists:", VRP_PANEL_PARQUET.exists())
print("VIX term structure exists:", VIX_TERM_STRUCTURE_PARQUET.exists())

# ----------------------------
# Load VRP panel
# ----------------------------

vrp_panel_df = pd.read_parquet(VRP_PANEL_PARQUET).copy()

print("\nLoaded VRP panel rows:", len(vrp_panel_df))
print("\nVRP panel columns:")
for c in vrp_panel_df.columns:
    print(" ", c)

# ----------------------------
# Helper: find tenor column
# ----------------------------

def find_tenor_column(df, expected_tenors):
    """
    Find a tenor column either by name or by values.
    """
    expected_set = set(expected_tenors)

    obvious_names = [
        "tenor",
        "target_tenor",
        "target_tenor_days",
        "tenor_days",
        "calendar_tenor",
        "calendar_tenor_days",
        "target_days",
        "target_dte",
        "dte",
        "days_to_expiry",
        "days_to_expiration",
        "expiry_days",
        "vix_tenor",
        "dte_target",
        "days",
    ]

    for col in obvious_names:
        if col in df.columns:
            return col

    value_match_cols = []

    for col in df.columns:
        series_numeric = pd.to_numeric(df[col], errors="coerce")
        unique_values = sorted(series_numeric.dropna().unique())

        if len(unique_values) <= 25:
            integer_like_values = []

            for x in unique_values:
                if float(x).is_integer():
                    integer_like_values.append(int(x))

            if set(integer_like_values) == expected_set:
                value_match_cols.append(col)

    if len(value_match_cols) >= 1:
        return value_match_cols[0]

    return None


# ----------------------------
# Standardize / recover tenor
# ----------------------------

found_tenor_col = find_tenor_column(vrp_panel_df, TENORS)

if found_tenor_col is not None:
    if found_tenor_col != "tenor":
        print(f"\nRenaming tenor column: {found_tenor_col} -> tenor")
        vrp_panel_df = vrp_panel_df.rename(columns={found_tenor_col: "tenor"})
    else:
        print("\nTenor column found directly: tenor")

else:
    print("\nNo tenor column found directly in VRP panel.")
    print("Attempting to recover tenor from repaired VIX term-structure panel...")

    if not VIX_TERM_STRUCTURE_PARQUET.exists():
        raise FileNotFoundError(
            f"Cannot recover tenor because VIX term-structure file is missing: "
            f"{VIX_TERM_STRUCTURE_PARQUET}"
        )

    vix_panel_df = pd.read_parquet(VIX_TERM_STRUCTURE_PARQUET).copy()

    print("\nLoaded VIX panel rows:", len(vix_panel_df))
    print("VIX panel columns:")
    for c in vix_panel_df.columns:
        print(" ", c)

    vix_tenor_col = find_tenor_column(vix_panel_df, TENORS)

    if vix_tenor_col is None:
        raise ValueError(
            "Could not find tenor column in VIX panel either. "
            "Paste the printed VIX panel columns."
        )

    if vix_tenor_col != "tenor":
        print(f"\nRenaming VIX tenor column: {vix_tenor_col} -> tenor")
        vix_panel_df = vix_panel_df.rename(columns={vix_tenor_col: "tenor"})

    required_vix_cols_for_recovery = [
        "trade_date",
        "tenor",
        "vix_style_vol",
    ]

    missing_vix_recovery_cols = [
        c for c in required_vix_cols_for_recovery
        if c not in vix_panel_df.columns
    ]

    if missing_vix_recovery_cols:
        raise ValueError(
            f"Missing VIX recovery columns: {missing_vix_recovery_cols}"
        )

    required_vrp_recovery_cols = [
        "trade_date",
        "vix_style_vol",
    ]

    missing_vrp_recovery_cols = [
        c for c in required_vrp_recovery_cols
        if c not in vrp_panel_df.columns
    ]

    if missing_vrp_recovery_cols:
        raise ValueError(
            f"Cannot recover tenor. Missing VRP recovery columns: {missing_vrp_recovery_cols}"
        )

    # Recover tenor by matching trade_date and rounded vix_style_vol.
    vrp_panel_df["trade_date"] = vrp_panel_df["trade_date"].astype(int)
    vix_panel_df["trade_date"] = vix_panel_df["trade_date"].astype(int)

    vrp_panel_df["_vix_style_vol_key"] = (
        pd.to_numeric(vrp_panel_df["vix_style_vol"], errors="coerce").round(10)
    )

    vix_recovery_df = vix_panel_df[
        [
            "trade_date",
            "tenor",
            "vix_style_vol",
        ]
    ].copy()

    vix_recovery_df["_vix_style_vol_key"] = (
        pd.to_numeric(vix_recovery_df["vix_style_vol"], errors="coerce").round(10)
    )

    vix_recovery_df = vix_recovery_df[
        [
            "trade_date",
            "_vix_style_vol_key",
            "tenor",
        ]
    ].drop_duplicates()

    before_rows = len(vrp_panel_df)

    vrp_panel_df = vrp_panel_df.merge(
        vix_recovery_df,
        on=[
            "trade_date",
            "_vix_style_vol_key",
        ],
        how="left",
        validate="many_to_one",
    )

    after_rows = len(vrp_panel_df)

    if before_rows != after_rows:
        raise ValueError(
            f"Tenor recovery changed row count from {before_rows} to {after_rows}."
        )

    missing_recovered_tenor = vrp_panel_df["tenor"].isna().sum()

    print("Recovered tenor missing rows:", missing_recovered_tenor)

    if missing_recovered_tenor > 0:
        display(
            vrp_panel_df[
                vrp_panel_df["tenor"].isna()
            ][
                [
                    "trade_date",
                    "vix_style_vol",
                    "_vix_style_vol_key",
                ]
            ].head(20)
        )

        raise ValueError(
            "Some tenors could not be recovered from VIX term-structure panel."
        )

    vrp_panel_df = vrp_panel_df.drop(columns=["_vix_style_vol_key"])

# ----------------------------
# Validate required columns
# ----------------------------

required_cols = [
    "trade_date",
    "tenor",
    "spx_close",
    "spx_log_return",
    "spx_rsi_14",
    "vix_style_vol",
    "trailing_realized_variance",
    "trailing_realized_vol",
    "primary_vrp_signal",
]

missing_cols = [
    c for c in required_cols
    if c not in vrp_panel_df.columns
]

if missing_cols:
    raise ValueError(f"Missing required columns from VRP panel: {missing_cols}")

vrp_panel_df["trade_date"] = vrp_panel_df["trade_date"].astype(int)
vrp_panel_df["tenor"] = vrp_panel_df["tenor"].astype(int)

vrp_panel_df = (
    vrp_panel_df
    .sort_values(["trade_date", "tenor"])
    .reset_index(drop=True)
)

print("\nVRP panel rows after tenor standardization:", len(vrp_panel_df))
print("Date range:", vrp_panel_df["trade_date"].min(), "to", vrp_panel_df["trade_date"].max())
print("Tenors:", sorted(vrp_panel_df["tenor"].unique()))

print("\nRows by tenor:")
display(
    vrp_panel_df["tenor"]
    .value_counts()
    .sort_index()
    .rename("rows")
    .to_frame()
)

# ----------------------------
# Build unique SPX return table
# ----------------------------

spx_df = (
    vrp_panel_df[
        [
            "trade_date",
            "spx_close",
            "spx_log_return",
            "spx_rsi_14",
        ]
    ]
    .drop_duplicates(subset=["trade_date"])
    .copy()
    .sort_values("trade_date")
    .reset_index(drop=True)
)

spx_df["trade_date_ts"] = pd.to_datetime(
    spx_df["trade_date"].astype(str),
    format="%Y%m%d",
)

spx_df["squared_log_return"] = spx_df["spx_log_return"] ** 2

print("\nSPX rows:", len(spx_df))
print("SPX date range:", spx_df["trade_date"].min(), "to", spx_df["trade_date"].max())
print("Missing SPX close:", spx_df["spx_close"].isna().sum())
print("Missing log returns:", spx_df["spx_log_return"].isna().sum())

display(spx_df.head())
display(spx_df.tail())

Project root: C:\Users\patri\vrp_project
VRP panel exists: True
VIX term structure exists: True

Loaded VRP panel rows: 18099

VRP panel columns:
  trade_date
  target_days
  rate_symbol
  rate_pct
  implied_variance
  vix_style_vol
  near_root
  near_expiration
  near_days
  near_variance
  next_root
  next_expiration
  next_days
  next_variance
  methodology_version
  expiration_universe
  trading_calendar_source
  rate_source
  option_source
  quote_time
  calc_time_ms
  market_close_time
  quote_time_used
  quote_time_offset_minutes
  raw_implied_variance
  raw_vix_style_vol
  is_repaired
  repair_method
  source_methodology_version
  trailing_realized_variance
  trailing_realized_vol
  trailing_realized_num_returns
  trailing_realized_window_complete
  forward_realized_variance
  forward_realized_vol
  forward_realized_num_returns
  forward_realized_window_complete
  vrp_trailing_variance
  vrp_trailing_vol_points
  vrp_trailing_variance_ratio
  vrp_trailing_log_variance_ratio
  v

,rows
tenor,
9,2011
12,2011
15,2011
18,2011
21,2011
24,2011
27,2011
30,2011
33,2011



SPX rows: 2011
SPX date range: 20180625 to 20260625
Missing SPX close: 0
Missing log returns: 0


,trade_date,spx_close,spx_log_return,spx_rsi_14,trade_date_ts,squared_log_return
0,20180625,2717.07,-0.013820,41.761584,2018-06-25,1.909866e-04
1,20180626,2723.06,0.002202,43.759005,2018-06-26,4.849483e-06
2,20180627,2699.63,-0.008642,38.235030,2018-06-27,7.467589e-05
3,20180628,2716.31,0.006160,43.685260,2018-06-28,3.794086e-05
4,20180629,2718.37,0.000758,44.338515,2018-06-29,5.747070e-07


,trade_date,spx_close,spx_log_return,spx_rsi_14,trade_date_ts,squared_log_return
2006,20260618,7500.58,0.010788,55.071587,2026-06-18,1.163770e-04
2007,20260622,7472.79,-0.003712,53.235432,2026-06-22,1.377841e-05
2008,20260623,7365.46,-0.014467,46.752076,2026-06-23,2.092917e-04
2009,20260624,7358.22,-0.000983,46.342083,2026-06-24,9.671736e-07
2010,20260625,7357.49,-0.000099,46.297996,2026-06-25,9.843358e-09


In [6]:
# ============================================================
# Cell 2: Realized variance helper
# ============================================================

def int_yyyymmdd_to_timestamp(date_int):
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d")


def calculate_calendar_window_realized_variance(
    spx_returns_df,
    anchor_date,
    window_days,
    direction,
):
    """
    Annualized realized variance using calendar-day windows.

    trailing:
        return_date > anchor_date - window_days
        return_date <= anchor_date

    forward:
        return_date > anchor_date
        return_date <= anchor_date + window_days

    Annualization:
        sum squared log returns / (window_days / 365)
    """
    anchor_ts = int_yyyymmdd_to_timestamp(anchor_date)

    if direction == "trailing":
        start_ts = anchor_ts - pd.Timedelta(days=int(window_days))
        end_ts = anchor_ts

        mask = (
            (spx_returns_df["trade_date_ts"] > start_ts)
            & (spx_returns_df["trade_date_ts"] <= end_ts)
        )

    elif direction == "forward":
        start_ts = anchor_ts
        end_ts = anchor_ts + pd.Timedelta(days=int(window_days))

        mask = (
            (spx_returns_df["trade_date_ts"] > start_ts)
            & (spx_returns_df["trade_date_ts"] <= end_ts)
        )

    else:
        raise ValueError("direction must be 'trailing' or 'forward'")

    window_returns = spx_returns_df.loc[mask, "squared_log_return"].dropna()

    if len(window_returns) == 0:
        return {
            "annualized_variance": np.nan,
            "annualized_vol": np.nan,
            "num_returns": 0,
        }

    variance_sum = window_returns.sum()
    annualized_variance = variance_sum / (float(window_days) / 365.0)
    annualized_vol = np.sqrt(annualized_variance) * 100.0

    return {
        "annualized_variance": annualized_variance,
        "annualized_vol": annualized_vol,
        "num_returns": len(window_returns),
    }


# Sanity test
test_date = spx_df["trade_date"].iloc[300]

print("Test date:", test_date)

print(
    "Trailing 30d:",
    calculate_calendar_window_realized_variance(
        spx_returns_df=spx_df,
        anchor_date=test_date,
        window_days=30,
        direction="trailing",
    )
)

print(
    "Forward 30d:",
    calculate_calendar_window_realized_variance(
        spx_returns_df=spx_df,
        anchor_date=test_date,
        window_days=30,
        direction="forward",
    )
)

Test date: 20190904
Trailing 30d: {'annualized_variance': np.float64(0.04227332259803317), 'annualized_vol': np.float64(20.560477279974112), 'num_returns': 21}
Forward 30d: {'annualized_variance': np.float64(0.014345241212439174), 'annualized_vol': np.float64(11.977162106458763), 'num_returns': 22}


In [7]:
# ============================================================
# Cell 3: Build forward targets and trailing realized-vol features
# ============================================================

base_dates = sorted(vrp_panel_df["trade_date"].unique())

panel_rows = []

for i, trade_date in enumerate(base_dates):
    if i % 250 == 0:
        print(f"Processing date {i}/{len(base_dates)}: {trade_date}")

    # ----------------------------
    # Date-level trailing features
    # ----------------------------

    date_feature_row = {
        "trade_date": int(trade_date),
    }

    for window in FEATURE_WINDOWS:
        trailing_result = calculate_calendar_window_realized_variance(
            spx_returns_df=spx_df,
            anchor_date=trade_date,
            window_days=window,
            direction="trailing",
        )

        trailing_var = trailing_result["annualized_variance"]

        date_feature_row[f"trailing_rv_{window}d_variance"] = trailing_var
        date_feature_row[f"trailing_rv_{window}d_vol"] = trailing_result["annualized_vol"]
        date_feature_row[f"trailing_rv_{window}d_num_returns"] = trailing_result["num_returns"]

        if pd.notna(trailing_var):
            date_feature_row[f"log_trailing_rv_{window}d_variance"] = np.log(
                max(trailing_var, EPSILON_VARIANCE)
            )
        else:
            date_feature_row[f"log_trailing_rv_{window}d_variance"] = np.nan

    # ----------------------------
    # Tenor-level forward targets
    # ----------------------------

    for tenor in TENORS:
        forward_result = calculate_calendar_window_realized_variance(
            spx_returns_df=spx_df,
            anchor_date=trade_date,
            window_days=tenor,
            direction="forward",
        )

        forward_var = forward_result["annualized_variance"]

        row = {
            **date_feature_row,
            "tenor": int(tenor),
            "forward_realized_variance": forward_var,
            "forward_realized_vol": forward_result["annualized_vol"],
            "forward_num_returns": forward_result["num_returns"],
        }

        if pd.notna(forward_var):
            row["log_forward_realized_variance"] = np.log(
                max(forward_var, EPSILON_VARIANCE)
            )
        else:
            row["log_forward_realized_variance"] = np.nan

        panel_rows.append(row)

feature_target_raw_df = pd.DataFrame(panel_rows)

print("Feature/target raw rows:", len(feature_target_raw_df))
print("Date range:", feature_target_raw_df["trade_date"].min(), "to", feature_target_raw_df["trade_date"].max())
print("Tenors:", sorted(feature_target_raw_df["tenor"].unique()))

print("\nMissing forward realized variance by tenor:")
display(
    feature_target_raw_df
    .groupby("tenor")["forward_realized_variance"]
    .apply(lambda x: x.isna().sum())
    .rename("missing_forward_realized_variance")
    .to_frame()
)

print("\nForward return counts by tenor:")
display(
    feature_target_raw_df
    .groupby("tenor")["forward_num_returns"]
    .describe()
)

display(feature_target_raw_df.tail(30))

Processing date 0/2011: 20180625
Processing date 250/2011: 20190624
Processing date 500/2011: 20200619
Processing date 750/2011: 20210617
Processing date 1000/2011: 20220614
Processing date 1250/2011: 20230613
Processing date 1500/2011: 20240611
Processing date 1750/2011: 20250611
Processing date 2000/2011: 20260610
Feature/target raw rows: 18099
Date range: 20180625 to 20260625
Tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]

Missing forward realized variance by tenor:


,missing_forward_realized_variance
tenor,
9,1
12,1
15,1
18,1
21,1
24,1
27,1
30,1
33,1



Forward return counts by tenor:


,count,mean,std,min,25%,50%,75%,max
tenor,,,,,,,,
9,2011.0,6.160617,0.918879,0.0,5.0,6.0,7.0,7.0
12,2011.0,7.858777,0.701063,0.0,7.0,8.0,8.0,9.0
15,2011.0,10.374441,0.808858,0.0,10.0,10.0,11.0,11.0
18,2011.0,12.069120,1.125329,0.0,11.0,12.0,13.0,14.0
21,2011.0,14.381402,0.962083,0.0,14.0,14.0,15.0,15.0
24,2011.0,16.291397,1.404903,0.0,16.0,16.0,17.0,18.0
27,2011.0,18.201392,1.241375,0.0,18.0,18.0,19.0,19.0
30,2011.0,20.505719,1.627605,0.0,20.0,21.0,21.0,22.0
33,2011.0,22.191944,1.626569,0.0,22.0,22.0,23.0,24.0


,trade_date,trailing_rv_5d_variance,trailing_rv_5d_vol,trailing_rv_5d_num_returns,log_trailing_rv_5d_variance,trailing_rv_9d_variance,trailing_rv_9d_vol,trailing_rv_9d_num_returns,log_trailing_rv_9d_variance,trailing_rv_12d_variance,...,log_trailing_rv_126d_variance,trailing_rv_252d_variance,trailing_rv_252d_vol,trailing_rv_252d_num_returns,log_trailing_rv_252d_variance,tenor,forward_realized_variance,forward_realized_vol,forward_num_returns,log_forward_realized_variance
18069,20260622,0.009501,9.747484,2,-4.656322,0.023554,15.347308,5,-3.748460,0.027604,...,-3.811142,0.017889,13.374905,172,-4.023580,27,2.842522e-03,5.331530,3,-5.863064
18070,20260622,0.009501,9.747484,2,-4.656322,0.023554,15.347308,5,-3.748460,0.027604,...,-3.811142,0.017889,13.374905,172,-4.023580,30,2.558270e-03,5.057934,3,-5.968424
18071,20260622,0.009501,9.747484,2,-4.656322,0.023554,15.347308,5,-3.748460,0.027604,...,-3.811142,0.017889,13.374905,172,-4.023580,33,2.325700e-03,4.822551,3,-6.063734
18072,20260623,0.016284,12.760924,2,-4.117565,0.032042,17.900259,6,-3.440710,0.024796,...,-3.784241,0.018188,13.486437,172,-4.006971,9,3.962347e-05,0.629472,2,-10.136089
18073,20260623,0.016284,12.760924,2,-4.117565,0.032042,17.900259,6,-3.440710,0.024796,...,-3.784241,0.018188,13.486437,172,-4.006971,12,2.971760e-05,0.545139,2,-10.423771
18074,20260623,0.016284,12.760924,2,-4.117565,0.032042,17.900259,6,-3.440710,0.024796,...,-3.784241,0.018188,13.486437,172,-4.006971,15,2.377408e-05,0.487587,2,-10.646915
18075,20260623,0.016284,12.760924,2,-4.117565,0.032042,17.900259,6,-3.440710,0.024796,...,-3.784241,0.018188,13.486437,172,-4.006971,18,1.981173e-05,0.445104,2,-10.829236
18076,20260623,0.016284,12.760924,2,-4.117565,0.032042,17.900259,6,-3.440710,0.024796,...,-3.784241,0.018188,13.486437,172,-4.006971,21,1.698149e-05,0.412086,2,-10.983387
18077,20260623,0.016284,12.760924,2,-4.117565,0.032042,17.900259,6,-3.440710,0.024796,...,-3.784241,0.018188,13.486437,172,-4.006971,24,1.485880e-05,0.385471,2,-11.116918
18078,20260623,0.016284,12.760924,2,-4.117565,0.032042,17.900259,6,-3.440710,0.024796,...,-3.784241,0.018188,13.486437,172,-4.006971,27,1.320782e-05,0.363426,2,-11.234701


In [8]:
# ============================================================
# Cell 4: Merge feature/target panel with existing VRP panel
# ============================================================

base_cols = [
    "trade_date",
    "tenor",
    "spx_close",
    "spx_log_return",
    "spx_rsi_14",
    "vix_style_vol",
    "trailing_realized_variance",
    "trailing_realized_vol",
    "vrp_trailing_variance_ratio",
    "vrp_trailing_log_variance_ratio",
    "vrp_trailing_vol_ratio",
    "primary_vrp_signal",
]

available_base_cols = [c for c in base_cols if c in vrp_panel_df.columns]

forecast_feature_target_df = (
    vrp_panel_df[available_base_cols]
    .copy()
    .merge(
        feature_target_raw_df,
        on=["trade_date", "tenor"],
        how="left",
        validate="one_to_one",
    )
)

forecast_feature_target_df["implied_variance"] = (
    forecast_feature_target_df["vix_style_vol"] / 100.0
) ** 2

forecast_feature_target_df["log_implied_variance"] = np.log(
    forecast_feature_target_df["implied_variance"].clip(lower=EPSILON_VARIANCE)
)

forecast_feature_target_df["log_trailing_realized_variance"] = np.log(
    forecast_feature_target_df["trailing_realized_variance"].clip(lower=EPSILON_VARIANCE)
)

print("Final feature/target rows:", len(forecast_feature_target_df))
print("Date range:", forecast_feature_target_df["trade_date"].min(), "to", forecast_feature_target_df["trade_date"].max())
print("Tenors:", sorted(forecast_feature_target_df["tenor"].unique()))

print("\nMissing key fields:")
key_fields = [
    "vix_style_vol",
    "trailing_realized_variance",
    "forward_realized_variance",
    "implied_variance",
    "log_forward_realized_variance",
]

display(
    forecast_feature_target_df[key_fields]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(forecast_feature_target_df.tail(30))

Final feature/target rows: 18099
Date range: 20180625 to 20260625
Tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]

Missing key fields:


,missing_count
vix_style_vol,0
trailing_realized_variance,0
forward_realized_variance,9
implied_variance,0
log_forward_realized_variance,9


,trade_date,tenor,spx_close,spx_log_return,spx_rsi_14,vix_style_vol,trailing_realized_variance,trailing_realized_vol,vrp_trailing_variance_ratio,vrp_trailing_log_variance_ratio,...,trailing_rv_252d_vol,trailing_rv_252d_num_returns,log_trailing_rv_252d_variance,forward_realized_variance,forward_realized_vol,forward_num_returns,log_forward_realized_variance,implied_variance,log_implied_variance,log_trailing_realized_variance
18069,20260622,27,7472.79,-0.003712,53.235432,17.159877,0.027380,16.547043,1.075443,0.072733,...,13.374905,172,-4.023580,2.842522e-03,5.331530,3,-5.863064,0.029446,-3.525193,-3.597926
18070,20260622,30,7472.79,-0.003712,53.235432,17.318125,0.025094,15.840962,1.195195,0.178309,...,13.374905,172,-4.023580,2.558270e-03,5.057934,3,-5.968424,0.029992,-3.506833,-3.685142
18071,20260622,33,7472.79,-0.003712,53.235432,17.502016,0.022998,15.165067,1.331949,0.286643,...,13.374905,172,-4.023580,2.325700e-03,4.822551,3,-6.063734,0.030632,-3.485708,-3.772351
18072,20260623,9,7365.46,-0.014467,46.752076,19.652659,0.032042,17.900259,1.205380,0.186795,...,13.486437,172,-4.006971,3.962347e-05,0.629472,2,-10.136089,0.038623,-3.253915,-3.440710
18073,20260623,12,7365.46,-0.014467,46.752076,19.402357,0.024796,15.746680,1.518207,0.417530,...,13.486437,172,-4.006971,2.971760e-05,0.545139,2,-10.423771,0.037645,-3.279551,-3.697081
18074,20260623,15,7365.46,-0.014467,46.752076,19.250614,0.033829,18.392573,1.095479,0.091192,...,13.486437,172,-4.006971,2.377408e-05,0.487587,2,-10.646915,0.037059,-3.295254,-3.386446
18075,20260623,18,7365.46,-0.014467,46.752076,19.231723,0.028370,16.843361,1.303704,0.265209,...,13.486437,172,-4.006971,1.981173e-05,0.445104,2,-10.829236,0.036986,-3.297218,-3.562427
18076,20260623,21,7365.46,-0.014467,46.752076,19.359673,0.038038,19.503400,0.985316,-0.014793,...,13.486437,172,-4.006971,1.698149e-05,0.412086,2,-10.983387,0.037480,-3.283956,-3.269163
18077,20260623,24,7365.46,-0.014467,46.752076,19.455023,0.033413,18.279328,1.132773,0.124669,...,13.486437,172,-4.006971,1.485880e-05,0.385471,2,-11.116918,0.037850,-3.274130,-3.398799
18078,20260623,27,7365.46,-0.014467,46.752076,19.528330,0.030209,17.380854,1.262374,0.232994,...,13.486437,172,-4.006971,1.320782e-05,0.363426,2,-11.234701,0.038136,-3.266608,-3.499602


In [9]:
# ============================================================
# Cell 5: Sanity check rebuilt trailing RV features
# ============================================================

comparison_rows = []

for tenor in TENORS:
    feature_col = f"trailing_rv_{tenor}d_variance"

    temp = forecast_feature_target_df[
        forecast_feature_target_df["tenor"] == tenor
    ].copy()

    temp = temp.dropna(
        subset=[
            "trailing_realized_variance",
            feature_col,
        ]
    )

    diff = temp[feature_col] - temp["trailing_realized_variance"]

    comparison_rows.append({
        "tenor": tenor,
        "rows": len(temp),
        "mean_diff": diff.mean(),
        "median_diff": diff.median(),
        "max_abs_diff": diff.abs().max(),
        "corr": temp[feature_col].corr(temp["trailing_realized_variance"]),
    })

trailing_rebuild_check_df = pd.DataFrame(comparison_rows)

display(trailing_rebuild_check_df)

print("Largest rebuild differences:")

largest_diff_rows = []

for tenor in TENORS:
    feature_col = f"trailing_rv_{tenor}d_variance"

    temp = forecast_feature_target_df[
        forecast_feature_target_df["tenor"] == tenor
    ].copy()

    temp["trailing_rebuild_abs_diff"] = (
        temp[feature_col] - temp["trailing_realized_variance"]
    ).abs()

    largest_diff_rows.append(
        temp.sort_values("trailing_rebuild_abs_diff", ascending=False)
        [
            [
                "trade_date",
                "tenor",
                "trailing_realized_variance",
                feature_col,
                "trailing_rebuild_abs_diff",
            ]
        ]
        .head(3)
    )

largest_diff_df = pd.concat(largest_diff_rows, ignore_index=True)

display(largest_diff_df)

,tenor,rows,mean_diff,median_diff,max_abs_diff,corr
0,9,2011,-0.000006,0.0,0.002745,1.000000
1,12,2011,-0.000006,0.0,0.002276,1.000000
2,15,2011,-0.000007,0.0,0.002318,0.999999
3,18,2011,-0.000008,0.0,0.002129,0.999999
4,21,2011,-0.000010,0.0,0.003108,0.999999
5,24,2011,-0.000012,0.0,0.003023,0.999999
6,27,2011,-0.000017,0.0,0.007057,0.999996
7,30,2011,-0.000026,0.0,0.007997,0.999992
8,33,2011,-0.000029,0.0,0.007377,0.999990


Largest rebuild differences:


,trade_date,tenor,trailing_realized_variance,trailing_rv_9d_variance,trailing_rebuild_abs_diff,trailing_rv_12d_variance,trailing_rv_15d_variance,trailing_rv_18d_variance,trailing_rv_21d_variance,trailing_rv_24d_variance,trailing_rv_27d_variance,trailing_rv_30d_variance,trailing_rv_33d_variance
0,20180625,9,0.010491,0.007746,0.002745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20180626,9,0.010688,0.007942,0.002745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20180627,9,0.013532,0.010971,0.002562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20180625,12,0.008085,NaN,0.002276,0.005809,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20180626,12,0.008047,NaN,0.002090,0.005957,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,20180629,12,0.011459,NaN,0.002059,0.009400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,20180625,15,0.006966,NaN,0.002318,NaN,0.004647,NaN,NaN,NaN,NaN,NaN,NaN
7,20180626,15,0.007056,NaN,0.002291,NaN,0.004765,NaN,NaN,NaN,NaN,NaN,NaN
8,20180627,15,0.008799,NaN,0.002217,NaN,0.006582,NaN,NaN,NaN,NaN,NaN,NaN
9,20180625,18,0.006002,NaN,0.002129,NaN,NaN,0.003873,NaN,NaN,NaN,NaN,NaN


In [10]:
# ============================================================
# Cell 6: Target coverage audit and save
# ============================================================

target_coverage_df = (
    forecast_feature_target_df
    .groupby("tenor")
    .agg(
        rows=("trade_date", "count"),
        missing_forward_variance=("forward_realized_variance", lambda x: x.isna().sum()),
        min_forward_num_returns=("forward_num_returns", "min"),
        median_forward_num_returns=("forward_num_returns", "median"),
        max_forward_num_returns=("forward_num_returns", "max"),
        first_date=("trade_date", "min"),
        last_date=("trade_date", "max"),
    )
    .reset_index()
)

target_coverage_df["usable_forward_target_rows"] = (
    target_coverage_df["rows"] - target_coverage_df["missing_forward_variance"]
)

target_coverage_df["missing_forward_target_pct"] = (
    target_coverage_df["missing_forward_variance"] / target_coverage_df["rows"]
)

display(target_coverage_df)

forecast_feature_target_df.to_parquet(
    VOL_FORECAST_FEATURE_TARGET_PARQUET,
    index=False,
)

target_coverage_df.to_csv(
    VOL_FORECAST_FEATURE_TARGET_AUDIT_CSV,
    index=False,
)

print("Saved feature/target panel:", VOL_FORECAST_FEATURE_TARGET_PARQUET)
print("Saved target audit:", VOL_FORECAST_FEATURE_TARGET_AUDIT_CSV)

,tenor,rows,missing_forward_variance,min_forward_num_returns,median_forward_num_returns,max_forward_num_returns,first_date,last_date,usable_forward_target_rows,missing_forward_target_pct
0,9,2011,1,0,6.0,7,20180625,20260625,2010,0.000497
1,12,2011,1,0,8.0,9,20180625,20260625,2010,0.000497
2,15,2011,1,0,10.0,11,20180625,20260625,2010,0.000497
3,18,2011,1,0,12.0,14,20180625,20260625,2010,0.000497
4,21,2011,1,0,14.0,15,20180625,20260625,2010,0.000497
5,24,2011,1,0,16.0,18,20180625,20260625,2010,0.000497
6,27,2011,1,0,18.0,19,20180625,20260625,2010,0.000497
7,30,2011,1,0,21.0,22,20180625,20260625,2010,0.000497
8,33,2011,1,0,22.0,24,20180625,20260625,2010,0.000497


Saved feature/target panel: C:\Users\patri\vrp_project\data\processed\vol_forecast_feature_target_panel_v0_1.parquet
Saved target audit: C:\Users\patri\vrp_project\data\audit\vol_forecast_feature_target_audit_v0_1.csv


In [11]:
# ============================================================
# Cell 7: EWMA variance forecasts
# ============================================================

EWMA_HALFLIVES = [10, 21, 63]

VOL_FORECAST_EWMA_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "vol_forecast_ewma_panel_v0_1.parquet"
)

VOL_FORECAST_EWMA_COMPARISON_CSV = (
    AUDIT_DIR / "vol_forecast_ewma_comparison_v0_1.csv"
)

VOL_FORECAST_EWMA_COMPARISON_BY_TENOR_CSV = (
    AUDIT_DIR / "vol_forecast_ewma_comparison_by_tenor_v0_1.csv"
)

# ----------------------------
# Build date-level EWMA daily variance estimates
# ----------------------------

ewma_daily_df = (
    spx_df[
        [
            "trade_date",
            "squared_log_return",
        ]
    ]
    .copy()
    .sort_values("trade_date")
    .reset_index(drop=True)
)

for half_life in EWMA_HALFLIVES:
    ewma_daily_df[f"ewma_{half_life}_daily_variance"] = (
        ewma_daily_df["squared_log_return"]
        .ewm(
            halflife=half_life,
            adjust=False,
            min_periods=half_life,
        )
        .mean()
    )

    ewma_daily_df[f"ewma_{half_life}_daily_vol_252"] = (
        np.sqrt(ewma_daily_df[f"ewma_{half_life}_daily_variance"] * 252.0) * 100.0
    )

display(ewma_daily_df.tail())

# ----------------------------
# Merge EWMA daily variance into feature/target panel
# ----------------------------

ewma_cols_to_drop = [
    c for c in forecast_feature_target_df.columns
    if c.startswith("ewma_")
]

if ewma_cols_to_drop:
    forecast_feature_target_df = forecast_feature_target_df.drop(columns=ewma_cols_to_drop)

forecast_ewma_df = forecast_feature_target_df.merge(
    ewma_daily_df.drop(columns=["squared_log_return"]),
    on="trade_date",
    how="left",
    validate="many_to_one",
)

# ----------------------------
# Convert daily EWMA variance into same-tenor annualized forecast variance
# ----------------------------

forecast_num_returns = forecast_ewma_df["forward_num_returns"].replace(0, np.nan)
tenor_year_fraction = forecast_ewma_df["tenor"] / 365.0

for half_life in EWMA_HALFLIVES:
    daily_var_col = f"ewma_{half_life}_daily_variance"
    forecast_var_col = f"ewma_{half_life}_forecast_variance"
    forecast_vol_col = f"ewma_{half_life}_forecast_vol"
    log_forecast_var_col = f"log_ewma_{half_life}_forecast_variance"

    forecast_ewma_df[forecast_var_col] = (
        forecast_ewma_df[daily_var_col] * forecast_num_returns / tenor_year_fraction
    )

    forecast_ewma_df[forecast_vol_col] = (
        np.sqrt(forecast_ewma_df[forecast_var_col]) * 100.0
    )

    forecast_ewma_df[log_forecast_var_col] = np.log(
        forecast_ewma_df[forecast_var_col].clip(lower=EPSILON_VARIANCE)
    )

# ----------------------------
# Blend EWMA forecasts in log-variance space
# ----------------------------

ewma_forecast_var_cols = [
    f"ewma_{half_life}_forecast_variance"
    for half_life in EWMA_HALFLIVES
]

ewma_log_forecast_var_cols = [
    f"log_ewma_{half_life}_forecast_variance"
    for half_life in EWMA_HALFLIVES
]

valid_ewma_count = forecast_ewma_df[ewma_forecast_var_cols].notna().sum(axis=1)

forecast_ewma_df["ewma_blend_forecast_variance"] = np.where(
    valid_ewma_count == len(EWMA_HALFLIVES),
    np.exp(forecast_ewma_df[ewma_log_forecast_var_cols].mean(axis=1)),
    np.nan,
)

forecast_ewma_df["ewma_blend_forecast_vol"] = (
    np.sqrt(forecast_ewma_df["ewma_blend_forecast_variance"]) * 100.0
)

forecast_ewma_df["log_ewma_blend_forecast_variance"] = np.log(
    forecast_ewma_df["ewma_blend_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

# ----------------------------
# EWMA-based VRP signal
# ----------------------------

forecast_ewma_df["ewma_blend_vrp_signal"] = np.log(
    forecast_ewma_df["implied_variance"].clip(lower=EPSILON_VARIANCE)
    / forecast_ewma_df["ewma_blend_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

forecast_ewma_df["ewma_blend_vrp_vol_ratio"] = (
    forecast_ewma_df["vix_style_vol"] / forecast_ewma_df["ewma_blend_forecast_vol"]
)

print("EWMA panel rows:", len(forecast_ewma_df))
print("Date range:", forecast_ewma_df["trade_date"].min(), "to", forecast_ewma_df["trade_date"].max())
print("Tenors:", sorted(forecast_ewma_df["tenor"].unique()))

print("\nMissing EWMA forecast variance fields:")
display(
    forecast_ewma_df[
        ewma_forecast_var_cols + ["ewma_blend_forecast_variance"]
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(forecast_ewma_df.tail(30))

,trade_date,squared_log_return,ewma_10_daily_variance,ewma_10_daily_vol_252,ewma_21_daily_variance,ewma_21_daily_vol_252,ewma_63_daily_variance,ewma_63_daily_vol_252
2006,20260618,1.163770e-04,0.000109,16.566826,0.000094,15.379867,0.000085,14.656815
2007,20260622,1.377841e-05,0.000103,16.074986,0.000091,15.165343,0.000084,14.589433
2008,20260623,2.092917e-04,0.000110,16.625882,0.000095,15.480457,0.000086,14.706921
2009,20260624,9.671736e-07,0.000102,16.064624,0.000092,15.229670,0.000085,14.627149
2010,20260625,9.843358e-09,0.000096,15.517458,0.000089,14.980417,0.000084,14.546913


EWMA panel rows: 18099
Date range: 20180625 to 20260625
Tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]

Missing EWMA forecast variance fields:


,missing_count
ewma_10_forecast_variance,90
ewma_21_forecast_variance,189
ewma_63_forecast_variance,567
ewma_blend_forecast_variance,567


,trade_date,tenor,spx_close,spx_log_return,spx_rsi_14,vix_style_vol,trailing_realized_variance,trailing_realized_vol,vrp_trailing_variance_ratio,vrp_trailing_log_variance_ratio,...,ewma_21_forecast_vol,log_ewma_21_forecast_variance,ewma_63_forecast_variance,ewma_63_forecast_vol,log_ewma_63_forecast_variance,ewma_blend_forecast_variance,ewma_blend_forecast_vol,log_ewma_blend_forecast_variance,ewma_blend_vrp_signal,ewma_blend_vrp_vol_ratio
18069,20260622,27,7472.79,-0.003712,53.235432,17.159877,0.027380,16.547043,1.075443,0.072733,...,6.083831,-5.599071,0.003426,5.852795,-5.676502,0.003750,6.123579,-5.586047,2.060854,2.802263
18070,20260622,30,7472.79,-0.003712,53.235432,17.318125,0.025094,15.840962,1.195195,0.178309,...,5.771629,-5.704432,0.003083,5.552449,-5.781862,0.003375,5.809337,-5.691407,2.184574,2.981084
18071,20260622,33,7472.79,-0.003712,53.235432,17.502016,0.022998,15.165067,1.331949,0.286643,...,5.503032,-5.799742,0.002803,5.294052,-5.877172,0.003068,5.538986,-5.786718,2.301009,3.159787
18072,20260623,9,7365.46,-0.014467,46.752076,19.652659,0.032042,17.900259,1.205380,0.186795,...,8.782612,-4.864793,0.006962,8.343757,-4.967313,0.007818,8.841718,-4.851378,1.597463,2.222720
18073,20260623,12,7365.46,-0.014467,46.752076,19.402357,0.024796,15.746680,1.518207,0.417530,...,7.605965,-5.152475,0.005221,7.225906,-5.254995,0.005863,7.657152,-5.139060,1.859509,2.533887
18074,20260623,15,7365.46,-0.014467,46.752076,19.250614,0.033829,18.392573,1.095479,0.091192,...,6.802982,-5.375618,0.004177,6.463047,-5.478139,0.004691,6.848765,-5.362204,2.066949,2.810815
18075,20260623,18,7365.46,-0.014467,46.752076,19.231723,0.028370,16.843361,1.303704,0.265209,...,6.210244,-5.557940,0.003481,5.899927,-5.660460,0.003909,6.252039,-5.544525,2.247307,3.076072
18076,20260623,21,7365.46,-0.014467,46.752076,19.359673,0.038038,19.503400,0.985316,-0.014793,...,5.749569,-5.712091,0.002984,5.462271,-5.814611,0.003350,5.788263,-5.698676,2.414720,3.344643
18077,20260623,24,7365.46,-0.014467,46.752076,19.455023,0.033413,18.279328,1.132773,0.124669,...,5.378229,-5.845622,0.002611,5.109487,-5.948142,0.002932,5.414424,-5.832207,2.558077,3.593184
18078,20260623,27,7365.46,-0.014467,46.752076,19.528330,0.030209,17.380854,1.262374,0.232994,...,5.070643,-5.963405,0.002321,4.817270,-6.065925,0.002606,5.104768,-5.949990,2.683382,3.825508


In [12]:
# ============================================================
# Cell 8: Forecast evaluation helper
# ============================================================

def evaluate_variance_forecast(
    df,
    model_name,
    forecast_variance_col,
    group_cols=None,
):
    """
    Evaluate a variance forecast versus forward realized variance.

    Metrics are mostly in log-variance space because variance is positive,
    skewed, and multiplicative.
    """
    if group_cols is None:
        group_cols = []

    temp = df.copy()

    temp = temp[
        temp[forecast_variance_col].notna()
        & temp["forward_realized_variance"].notna()
        & (temp[forecast_variance_col] > 0)
        & (temp["forward_realized_variance"] > 0)
    ].copy()

    temp["log_forecast_variance"] = np.log(
        temp[forecast_variance_col].clip(lower=EPSILON_VARIANCE)
    )

    temp["log_forward_realized_variance"] = np.log(
        temp["forward_realized_variance"].clip(lower=EPSILON_VARIANCE)
    )

    temp["log_error"] = (
        temp["log_forecast_variance"]
        - temp["log_forward_realized_variance"]
    )

    temp["abs_log_error"] = temp["log_error"].abs()
    temp["squared_log_error"] = temp["log_error"] ** 2

    temp["forecast_to_realized_variance_ratio"] = (
        temp[forecast_variance_col] / temp["forward_realized_variance"]
    )

    def summarize_one_group(x):
        return pd.Series({
            "rows": len(x),
            "bias_log_forecast_minus_realized": x["log_error"].mean(),
            "mae_log": x["abs_log_error"].mean(),
            "rmse_log": np.sqrt(x["squared_log_error"].mean()),
            "median_abs_log_error": x["abs_log_error"].median(),
            "corr_log_forecast_vs_realized": x["log_forecast_variance"].corr(
                x["log_forward_realized_variance"]
            ),
            "avg_forecast_to_realized_variance_ratio": (
                x["forecast_to_realized_variance_ratio"].mean()
            ),
            "median_forecast_to_realized_variance_ratio": (
                x["forecast_to_realized_variance_ratio"].median()
            ),
            "avg_forward_realized_vol": x["forward_realized_vol"].mean(),
            "avg_forecast_vol": (np.sqrt(x[forecast_variance_col]) * 100.0).mean(),
        })

    if group_cols:
        out = (
            temp
            .groupby(group_cols)
            .apply(summarize_one_group)
            .reset_index()
        )
    else:
        out = summarize_one_group(temp).to_frame().T

    out.insert(0, "model", model_name)

    return out


forecast_model_specs = [
    {
        "model": "trailing_realized_variance_baseline",
        "forecast_variance_col": "trailing_realized_variance",
    },
    {
        "model": "ewma_10",
        "forecast_variance_col": "ewma_10_forecast_variance",
    },
    {
        "model": "ewma_21",
        "forecast_variance_col": "ewma_21_forecast_variance",
    },
    {
        "model": "ewma_63",
        "forecast_variance_col": "ewma_63_forecast_variance",
    },
    {
        "model": "ewma_blend",
        "forecast_variance_col": "ewma_blend_forecast_variance",
    },
]

In [13]:
# ============================================================
# Cell 9: Evaluate EWMA forecasts versus forward realized variance
# ============================================================

overall_eval_rows = []
tenor_eval_rows = []

for spec in forecast_model_specs:
    overall_eval_rows.append(
        evaluate_variance_forecast(
            df=forecast_ewma_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=[],
        )
    )

    tenor_eval_rows.append(
        evaluate_variance_forecast(
            df=forecast_ewma_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=["tenor"],
        )
    )

forecast_comparison_df = pd.concat(
    overall_eval_rows,
    ignore_index=True,
)

forecast_comparison_by_tenor_df = pd.concat(
    tenor_eval_rows,
    ignore_index=True,
)

print("Overall forecast comparison:")
display(
    forecast_comparison_df.sort_values("rmse_log")
)

print("Forecast comparison by tenor:")
display(
    forecast_comparison_by_tenor_df.sort_values(["tenor", "rmse_log"])
)

print("Front-tenor comparison only:")
display(
    forecast_comparison_by_tenor_df[
        forecast_comparison_by_tenor_df["tenor"].isin([9, 12, 15, 18])
    ].sort_values(["tenor", "rmse_log"])
)

C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the gr

Overall forecast comparison:


C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_group)


,model,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
1,ewma_10,18009.0,0.152984,0.707884,0.928618,0.564315,0.566777,6.954364,1.200741,15.933831,16.573582
4,ewma_blend,17532.0,0.275676,0.730246,0.956434,0.588256,0.518424,6.805920,1.350454,16.165649,17.311027
2,ewma_21,17910.0,0.266273,0.751932,0.982969,0.596655,0.507983,6.702096,1.328376,15.980386,17.220299
0,trailing_realized_variance_baseline,18090.0,0.033139,0.744142,0.999440,0.595182,0.534185,87.655811,1.096107,15.902944,16.091999
3,ewma_63,17532.0,0.445356,0.849882,1.089258,0.709935,0.386658,6.871310,1.652578,16.165649,18.445003


Forecast comparison by tenor:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
9,ewma_10,9,2001.0,0.271966,0.771506,1.001762,0.659237,0.598392,7.306389,1.308569,15.556921,16.638580
36,ewma_blend,9,1948.0,0.393260,0.828410,1.064771,0.697617,0.544382,7.261827,1.448450,15.795828,17.376500
18,ewma_21,9,1990.0,0.385253,0.843754,1.082566,0.702478,0.535472,7.140622,1.455219,15.602620,17.285899
0,trailing_realized_variance_baseline,9,2010.0,0.083543,0.826110,1.091774,0.643395,0.556045,28.601153,1.134990,15.530672,16.063016
27,ewma_63,9,1948.0,0.562941,0.960348,1.223592,0.799264,0.404949,7.548548,1.755551,15.795828,18.512271
10,ewma_10,12,2001.0,0.230158,0.740443,0.952442,0.636422,0.595385,7.110918,1.298758,15.364099,16.301387
37,ewma_blend,12,1948.0,0.351063,0.788484,1.006508,0.658731,0.539840,7.013141,1.419740,15.601448,17.024477
19,ewma_21,12,1990.0,0.343587,0.804522,1.027600,0.667875,0.530877,6.905402,1.395841,15.408818,16.934291
1,trailing_realized_variance_baseline,12,2010.0,0.054021,0.777463,1.029848,0.627210,0.563261,42.003096,1.163674,15.338475,15.696962
28,ewma_63,12,1948.0,0.520743,0.912713,1.159839,0.767822,0.397507,7.181280,1.738783,15.601448,18.137447


Front-tenor comparison only:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
9,ewma_10,9,2001.0,0.271966,0.771506,1.001762,0.659237,0.598392,7.306389,1.308569,15.556921,16.638580
36,ewma_blend,9,1948.0,0.393260,0.828410,1.064771,0.697617,0.544382,7.261827,1.448450,15.795828,17.376500
18,ewma_21,9,1990.0,0.385253,0.843754,1.082566,0.702478,0.535472,7.140622,1.455219,15.602620,17.285899
0,trailing_realized_variance_baseline,9,2010.0,0.083543,0.826110,1.091774,0.643395,0.556045,28.601153,1.134990,15.530672,16.063016
27,ewma_63,9,1948.0,0.562941,0.960348,1.223592,0.799264,0.404949,7.548548,1.755551,15.795828,18.512271
10,ewma_10,12,2001.0,0.230158,0.740443,0.952442,0.636422,0.595385,7.110918,1.298758,15.364099,16.301387
37,ewma_blend,12,1948.0,0.351063,0.788484,1.006508,0.658731,0.539840,7.013141,1.419740,15.601448,17.024477
19,ewma_21,12,1990.0,0.343587,0.804522,1.027600,0.667875,0.530877,6.905402,1.395841,15.408818,16.934291
1,trailing_realized_variance_baseline,12,2010.0,0.054021,0.777463,1.029848,0.627210,0.563261,42.003096,1.163674,15.338475,15.696962
28,ewma_63,12,1948.0,0.520743,0.912713,1.159839,0.767822,0.397507,7.181280,1.738783,15.601448,18.137447


In [14]:
# ============================================================
# Cell 10: Save EWMA forecast panel and audit outputs
# ============================================================

forecast_ewma_df.to_parquet(
    VOL_FORECAST_EWMA_PANEL_PARQUET,
    index=False,
)

forecast_comparison_df.to_csv(
    VOL_FORECAST_EWMA_COMPARISON_CSV,
    index=False,
)

forecast_comparison_by_tenor_df.to_csv(
    VOL_FORECAST_EWMA_COMPARISON_BY_TENOR_CSV,
    index=False,
)

print("Saved EWMA forecast panel:", VOL_FORECAST_EWMA_PANEL_PARQUET)
print("Saved EWMA comparison:", VOL_FORECAST_EWMA_COMPARISON_CSV)
print("Saved EWMA comparison by tenor:", VOL_FORECAST_EWMA_COMPARISON_BY_TENOR_CSV)

Saved EWMA forecast panel: C:\Users\patri\vrp_project\data\processed\vol_forecast_ewma_panel_v0_1.parquet
Saved EWMA comparison: C:\Users\patri\vrp_project\data\audit\vol_forecast_ewma_comparison_v0_1.csv
Saved EWMA comparison by tenor: C:\Users\patri\vrp_project\data\audit\vol_forecast_ewma_comparison_by_tenor_v0_1.csv


In [15]:
# ============================================================
# Cell 11: Ex-ante bias-adjusted EWMA 10 forecast
# ============================================================

EWMA_BIAS_MIN_OBS = 252

VOL_FORECAST_EWMA_BIAS_ADJUSTED_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "vol_forecast_ewma_bias_adjusted_panel_v0_1.parquet"
)

VOL_FORECAST_EWMA_BIAS_COMPARISON_CSV = (
    AUDIT_DIR / "vol_forecast_ewma_bias_comparison_v0_1.csv"
)

VOL_FORECAST_EWMA_BIAS_COMPARISON_BY_TENOR_CSV = (
    AUDIT_DIR / "vol_forecast_ewma_bias_comparison_by_tenor_v0_1.csv"
)

VOL_FORECAST_EWMA_BIAS_COMPARISON_POST_WARMUP_CSV = (
    AUDIT_DIR / "vol_forecast_ewma_bias_comparison_post_warmup_v0_1.csv"
)

VOL_FORECAST_EWMA_BIAS_COMPARISON_BY_TENOR_POST_WARMUP_CSV = (
    AUDIT_DIR / "vol_forecast_ewma_bias_comparison_by_tenor_post_warmup_v0_1.csv"
)

bias_adjusted_df = forecast_ewma_df.copy()

# Timestamp versions for avoiding lookahead in the bias adjustment.
bias_adjusted_df["trade_date_ts"] = pd.to_datetime(
    bias_adjusted_df["trade_date"].astype(str),
    format="%Y%m%d",
)

bias_adjusted_df["target_end_date_ts"] = (
    bias_adjusted_df["trade_date_ts"]
    + pd.to_timedelta(bias_adjusted_df["tenor"], unit="D")
)

# Raw EWMA 10 log forecast error.
# Positive means realized variance exceeded forecast variance.
# Negative means EWMA forecast was too high.
bias_adjusted_df["ewma_10_realized_minus_forecast_log_error"] = (
    bias_adjusted_df["log_forward_realized_variance"]
    - bias_adjusted_df["log_ewma_10_forecast_variance"]
)

bias_adjusted_df["ewma_10_bias_adjustment_log"] = np.nan
bias_adjusted_df["ewma_10_bias_adjustment_obs"] = 0

# Build ex-ante bias correction by tenor.
# For each current date, only use prior rows whose target window has completed
# strictly before the current trade date.
for tenor in TENORS:
    tenor_mask = bias_adjusted_df["tenor"] == tenor

    tenor_df = (
        bias_adjusted_df.loc[
            tenor_mask,
            [
                "trade_date",
                "trade_date_ts",
                "target_end_date_ts",
                "ewma_10_realized_minus_forecast_log_error",
            ],
        ]
        .copy()
        .sort_values("trade_date_ts")
        .reset_index()
    )

    adjustments = []
    adjustment_obs = []

    for _, current_row in tenor_df.iterrows():
        current_trade_date_ts = current_row["trade_date_ts"]

        eligible_errors = tenor_df[
            (tenor_df["target_end_date_ts"] < current_trade_date_ts)
            & tenor_df["ewma_10_realized_minus_forecast_log_error"].notna()
        ]["ewma_10_realized_minus_forecast_log_error"]

        if len(eligible_errors) >= EWMA_BIAS_MIN_OBS:
            adjustments.append(eligible_errors.mean())
            adjustment_obs.append(len(eligible_errors))
        else:
            adjustments.append(np.nan)
            adjustment_obs.append(len(eligible_errors))

    tenor_df["ewma_10_bias_adjustment_log"] = adjustments
    tenor_df["ewma_10_bias_adjustment_obs"] = adjustment_obs

    bias_adjusted_df.loc[
        tenor_df["index"],
        "ewma_10_bias_adjustment_log",
    ] = tenor_df["ewma_10_bias_adjustment_log"].values

    bias_adjusted_df.loc[
        tenor_df["index"],
        "ewma_10_bias_adjustment_obs",
    ] = tenor_df["ewma_10_bias_adjustment_obs"].values

# Strict version: only exists after enough completed historical errors.
bias_adjusted_df["log_ewma_10_bias_adjusted_forecast_variance_strict"] = (
    bias_adjusted_df["log_ewma_10_forecast_variance"]
    + bias_adjusted_df["ewma_10_bias_adjustment_log"]
)

bias_adjusted_df["ewma_10_bias_adjusted_forecast_variance_strict"] = np.exp(
    bias_adjusted_df["log_ewma_10_bias_adjusted_forecast_variance_strict"]
)

bias_adjusted_df["ewma_10_bias_adjusted_forecast_vol_strict"] = (
    np.sqrt(bias_adjusted_df["ewma_10_bias_adjusted_forecast_variance_strict"])
    * 100.0
)

# Fallback version: use adjusted forecast when available, otherwise raw EWMA 10.
bias_adjusted_df["log_ewma_10_bias_adjusted_forecast_variance"] = (
    bias_adjusted_df["log_ewma_10_bias_adjusted_forecast_variance_strict"]
    .fillna(bias_adjusted_df["log_ewma_10_forecast_variance"])
)

bias_adjusted_df["ewma_10_bias_adjusted_forecast_variance"] = np.exp(
    bias_adjusted_df["log_ewma_10_bias_adjusted_forecast_variance"]
)

bias_adjusted_df["ewma_10_bias_adjusted_forecast_vol"] = (
    np.sqrt(bias_adjusted_df["ewma_10_bias_adjusted_forecast_variance"])
    * 100.0
)

bias_adjusted_df["ewma_10_bias_adjusted_vrp_signal"] = np.log(
    bias_adjusted_df["implied_variance"].clip(lower=EPSILON_VARIANCE)
    / bias_adjusted_df["ewma_10_bias_adjusted_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

bias_adjusted_df["ewma_10_bias_adjusted_vrp_vol_ratio"] = (
    bias_adjusted_df["vix_style_vol"]
    / bias_adjusted_df["ewma_10_bias_adjusted_forecast_vol"]
)

print("Bias-adjusted EWMA panel rows:", len(bias_adjusted_df))

print("\nBias adjustment availability by tenor:")
display(
    bias_adjusted_df
    .groupby("tenor")
    .agg(
        rows=("trade_date", "count"),
        strict_bias_adjusted_rows=("ewma_10_bias_adjusted_forecast_variance_strict", lambda x: x.notna().sum()),
        min_bias_obs=("ewma_10_bias_adjustment_obs", "min"),
        median_bias_obs=("ewma_10_bias_adjustment_obs", "median"),
        max_bias_obs=("ewma_10_bias_adjustment_obs", "max"),
        avg_bias_adjustment_log=("ewma_10_bias_adjustment_log", "mean"),
        median_bias_adjustment_log=("ewma_10_bias_adjustment_log", "median"),
    )
    .reset_index()
)

display(
    bias_adjusted_df[
        [
            "trade_date",
            "tenor",
            "ewma_10_forecast_vol",
            "ewma_10_bias_adjustment_log",
            "ewma_10_bias_adjusted_forecast_vol",
            "forward_realized_vol",
            "ewma_10_vrp_signal",
            "ewma_10_bias_adjusted_vrp_signal",
        ]
    ]
    .tail(30)
)

Bias-adjusted EWMA panel rows: 18099

Bias adjustment availability by tenor:


,tenor,rows,strict_bias_adjusted_rows,min_bias_obs,median_bias_obs,max_bias_obs,avg_bias_adjustment_log,median_bias_adjustment_log
0,9,2011,1742,0,990.0,1995,-0.317912,-0.300283
1,12,2011,1741,0,989.0,1994,-0.268288,-0.251597
2,15,2011,1738,0,986.0,1991,-0.222088,-0.206924
3,18,2011,1736,0,985.0,1989,-0.194287,-0.177261
4,21,2011,1734,0,982.0,1987,-0.164288,-0.150461
5,24,2011,1731,0,981.0,1984,-0.141028,-0.127336
6,27,2011,1730,0,979.0,1983,-0.122689,-0.112541
7,30,2011,1727,0,976.0,1980,-0.102179,-0.092959
8,33,2011,1726,0,975.0,1980,-0.088966,-0.082330


KeyError: "['ewma_10_vrp_signal'] not in index"

In [16]:
# ============================================================
# Patch after Cell 11:
# create raw EWMA 10 VRP signal and display final rows
# ============================================================

if "ewma_10_vrp_signal" not in bias_adjusted_df.columns:
    bias_adjusted_df["ewma_10_vrp_signal"] = np.log(
        bias_adjusted_df["implied_variance"].clip(lower=EPSILON_VARIANCE)
        / bias_adjusted_df["ewma_10_forecast_variance"].clip(lower=EPSILON_VARIANCE)
    )

if "ewma_10_vrp_vol_ratio" not in bias_adjusted_df.columns:
    bias_adjusted_df["ewma_10_vrp_vol_ratio"] = (
        bias_adjusted_df["vix_style_vol"]
        / bias_adjusted_df["ewma_10_forecast_vol"]
    )

display(
    bias_adjusted_df[
        [
            "trade_date",
            "tenor",
            "ewma_10_forecast_vol",
            "ewma_10_bias_adjustment_log",
            "ewma_10_bias_adjusted_forecast_vol",
            "forward_realized_vol",
            "ewma_10_vrp_signal",
            "ewma_10_bias_adjusted_vrp_signal",
        ]
    ]
    .tail(30)
)

,trade_date,tenor,ewma_10_forecast_vol,ewma_10_bias_adjustment_log,ewma_10_bias_adjusted_forecast_vol,forward_realized_vol,ewma_10_vrp_signal,ewma_10_bias_adjusted_vrp_signal
18069,20260622,27,6.448749,-0.102660,6.126087,5.331530,1.957375,2.060035
18070,20260622,30,6.117821,-0.082188,5.871512,5.057934,2.081095,2.163283
18071,20260622,33,5.833113,-0.073891,5.621540,4.822551,2.197530,2.271421
18072,20260623,9,9.432452,-0.264560,8.263729,0.629472,1.468113,1.732673
18073,20260623,12,8.168743,-0.222870,7.307344,0.545139,1.730159,1.953029
18074,20260623,15,7.306346,-0.181387,6.672868,0.487587,1.937599,2.118986
18075,20260623,18,6.669751,-0.154237,6.174722,0.445104,2.117957,2.272195
18076,20260623,21,6.174989,-0.136541,5.767487,0.412086,2.285370,2.421911
18077,20260623,24,5.776174,-0.112047,5.461471,0.385471,2.428728,2.540774
18078,20260623,27,5.445829,-0.102282,5.174326,0.363426,2.554033,2.656314


In [17]:
# ============================================================
# Cell 12: Evaluate bias-adjusted EWMA 10
# ============================================================

bias_forecast_model_specs = [
    {
        "model": "trailing_realized_variance_baseline",
        "forecast_variance_col": "trailing_realized_variance",
    },
    {
        "model": "ewma_10_raw",
        "forecast_variance_col": "ewma_10_forecast_variance",
    },
    {
        "model": "ewma_10_bias_adjusted_fallback",
        "forecast_variance_col": "ewma_10_bias_adjusted_forecast_variance",
    },
    {
        "model": "ewma_10_bias_adjusted_strict",
        "forecast_variance_col": "ewma_10_bias_adjusted_forecast_variance_strict",
    },
]

overall_bias_eval_rows = []
tenor_bias_eval_rows = []

for spec in bias_forecast_model_specs:
    overall_bias_eval_rows.append(
        evaluate_variance_forecast(
            df=bias_adjusted_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=[],
        )
    )

    tenor_bias_eval_rows.append(
        evaluate_variance_forecast(
            df=bias_adjusted_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=["tenor"],
        )
    )

bias_comparison_df = pd.concat(
    overall_bias_eval_rows,
    ignore_index=True,
)

bias_comparison_by_tenor_df = pd.concat(
    tenor_bias_eval_rows,
    ignore_index=True,
)

print("Overall comparison including bias-adjusted EWMA 10:")
display(
    bias_comparison_df.sort_values("rmse_log")
)

print("Comparison by tenor including bias-adjusted EWMA 10:")
display(
    bias_comparison_by_tenor_df.sort_values(["tenor", "rmse_log"])
)

print("Front-tenor comparison:")
display(
    bias_comparison_by_tenor_df[
        bias_comparison_by_tenor_df["tenor"].isin([9, 12, 15, 18])
    ].sort_values(["tenor", "rmse_log"])
)

C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the gr

Overall comparison including bias-adjusted EWMA 10:


,model,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
3,ewma_10_bias_adjusted_strict,15605.0,-0.034969,0.673818,0.901603,0.531868,0.588830,6.645651,0.982376,16.303430,15.579743
2,ewma_10_bias_adjusted_fallback,18009.0,-0.003367,0.701360,0.926073,0.559738,0.560702,6.009649,1.022650,15.933831,15.387235
1,ewma_10_raw,18009.0,0.152984,0.707884,0.928618,0.564315,0.566777,6.954364,1.200741,15.933831,16.573582
0,trailing_realized_variance_baseline,18090.0,0.033139,0.744142,0.999440,0.595182,0.534185,87.655811,1.096107,15.902944,16.091999


Comparison by tenor including bias-adjusted EWMA 10:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
27,ewma_10_bias_adjusted_strict,9,1742.0,-0.065011,0.734333,0.963242,0.585135,0.602348,6.105521,0.933536,15.966129,14.524413
18,ewma_10_bias_adjusted_fallback,9,2001.0,-0.004824,0.750776,0.979109,0.603674,0.585050,5.610804,0.997657,15.556921,14.494797
9,ewma_10_raw,9,2001.0,0.271966,0.771506,1.001762,0.659237,0.598392,7.306389,1.308569,15.556921,16.638580
0,trailing_realized_variance_baseline,9,2010.0,0.083543,0.826110,1.091774,0.643395,0.556045,28.601153,1.134990,15.530672,16.063016
28,ewma_10_bias_adjusted_strict,12,1741.0,-0.057069,0.703504,0.924414,0.586784,0.599223,6.236205,0.973706,15.772054,14.604556
19,ewma_10_bias_adjusted_fallback,12,2001.0,-0.003293,0.719569,0.938415,0.600919,0.583147,5.694865,1.041324,15.364099,14.527144
10,ewma_10_raw,12,2001.0,0.230158,0.740443,0.952442,0.636422,0.595385,7.110918,1.298758,15.364099,16.301387
1,trailing_realized_variance_baseline,12,2010.0,0.054021,0.777463,1.029848,0.627210,0.563261,42.003096,1.163674,15.338475,15.696962
29,ewma_10_bias_adjusted_strict,15,1738.0,-0.048542,0.684224,0.903909,0.572479,0.596647,6.435510,0.989614,16.361159,15.384996
20,ewma_10_bias_adjusted_fallback,15,2001.0,-0.004789,0.705342,0.920656,0.590906,0.576870,5.842334,1.036302,15.952008,15.245653


Front-tenor comparison:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
27,ewma_10_bias_adjusted_strict,9,1742.0,-0.065011,0.734333,0.963242,0.585135,0.602348,6.105521,0.933536,15.966129,14.524413
18,ewma_10_bias_adjusted_fallback,9,2001.0,-0.004824,0.750776,0.979109,0.603674,0.585050,5.610804,0.997657,15.556921,14.494797
9,ewma_10_raw,9,2001.0,0.271966,0.771506,1.001762,0.659237,0.598392,7.306389,1.308569,15.556921,16.638580
0,trailing_realized_variance_baseline,9,2010.0,0.083543,0.826110,1.091774,0.643395,0.556045,28.601153,1.134990,15.530672,16.063016
28,ewma_10_bias_adjusted_strict,12,1741.0,-0.057069,0.703504,0.924414,0.586784,0.599223,6.236205,0.973706,15.772054,14.604556
19,ewma_10_bias_adjusted_fallback,12,2001.0,-0.003293,0.719569,0.938415,0.600919,0.583147,5.694865,1.041324,15.364099,14.527144
10,ewma_10_raw,12,2001.0,0.230158,0.740443,0.952442,0.636422,0.595385,7.110918,1.298758,15.364099,16.301387
1,trailing_realized_variance_baseline,12,2010.0,0.054021,0.777463,1.029848,0.627210,0.563261,42.003096,1.163674,15.338475,15.696962
29,ewma_10_bias_adjusted_strict,15,1738.0,-0.048542,0.684224,0.903909,0.572479,0.596647,6.435510,0.989614,16.361159,15.384996
20,ewma_10_bias_adjusted_fallback,15,2001.0,-0.004789,0.705342,0.920656,0.590906,0.576870,5.842334,1.036302,15.952008,15.245653


In [18]:
# ============================================================
# Cell 13: Same-sample post-warmup comparison
# ============================================================

post_warmup_df = bias_adjusted_df[
    bias_adjusted_df["ewma_10_bias_adjustment_log"].notna()
].copy()

print("Post-warmup rows:", len(post_warmup_df))
print("Post-warmup date range:", post_warmup_df["trade_date"].min(), "to", post_warmup_df["trade_date"].max())

post_warmup_eval_rows = []
post_warmup_tenor_eval_rows = []

for spec in [
    {
        "model": "trailing_realized_variance_baseline",
        "forecast_variance_col": "trailing_realized_variance",
    },
    {
        "model": "ewma_10_raw",
        "forecast_variance_col": "ewma_10_forecast_variance",
    },
    {
        "model": "ewma_10_bias_adjusted",
        "forecast_variance_col": "ewma_10_bias_adjusted_forecast_variance",
    },
]:
    post_warmup_eval_rows.append(
        evaluate_variance_forecast(
            df=post_warmup_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=[],
        )
    )

    post_warmup_tenor_eval_rows.append(
        evaluate_variance_forecast(
            df=post_warmup_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=["tenor"],
        )
    )

post_warmup_comparison_df = pd.concat(
    post_warmup_eval_rows,
    ignore_index=True,
)

post_warmup_comparison_by_tenor_df = pd.concat(
    post_warmup_tenor_eval_rows,
    ignore_index=True,
)

print("Post-warmup same-sample overall comparison:")
display(
    post_warmup_comparison_df.sort_values("rmse_log")
)

print("Post-warmup same-sample by tenor:")
display(
    post_warmup_comparison_by_tenor_df.sort_values(["tenor", "rmse_log"])
)

print("Post-warmup front-tenor comparison:")
display(
    post_warmup_comparison_by_tenor_df[
        post_warmup_comparison_by_tenor_df["tenor"].isin([9, 12, 15, 18])
    ].sort_values(["tenor", "rmse_log"])
)

Post-warmup rows: 15614
Post-warmup date range: 20190719 to 20260625
Post-warmup same-sample overall comparison:


C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_one_group)
C:\Users\patri\AppData\Local\Temp\ipykernel_21960\219117882.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the gr

,model,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
2,ewma_10_bias_adjusted,15605.0,-0.034969,0.673818,0.901603,0.531868,0.588830,6.645651,0.982376,16.30343,15.579743
1,ewma_10_raw,15605.0,0.145469,0.681348,0.904619,0.536567,0.590858,7.735903,1.176780,16.30343,16.948852
0,trailing_realized_variance_baseline,15605.0,0.044294,0.734421,0.990456,0.590531,0.538563,101.375835,1.092695,16.30343,16.568702


Post-warmup same-sample by tenor:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
18,ewma_10_bias_adjusted,9,1742.0,-0.065011,0.734333,0.963242,0.585135,0.602348,6.105521,0.933536,15.966129,14.524413
9,ewma_10_raw,9,1742.0,0.252932,0.758145,0.989636,0.638577,0.604832,8.053205,1.270484,15.966129,16.986933
0,trailing_realized_variance_baseline,9,1742.0,0.084441,0.830584,1.098824,0.649170,0.546758,32.734106,1.128212,15.966129,16.512511
19,ewma_10_bias_adjusted,12,1741.0,-0.057069,0.703504,0.924414,0.586784,0.599223,6.236205,0.973706,15.772054,14.604556
10,ewma_10_raw,12,1741.0,0.211245,0.727495,0.940757,0.628195,0.602258,7.863730,1.260988,15.772054,16.643763
1,trailing_realized_variance_baseline,12,1741.0,0.053141,0.785446,1.040216,0.629956,0.550582,48.255231,1.159359,15.772054,16.129914
20,ewma_10_bias_adjusted,15,1738.0,-0.048542,0.684224,0.903909,0.572479,0.596647,6.435510,0.989614,16.361159,15.384996
11,ewma_10_raw,15,1738.0,0.173571,0.702894,0.912000,0.574457,0.600245,7.770278,1.236531,16.361159,17.123509
2,trailing_realized_variance_baseline,15,1738.0,0.031874,0.754674,1.007330,0.621250,0.544085,83.284308,1.147431,16.361159,16.530656
21,ewma_10_bias_adjusted,18,1736.0,-0.046050,0.674080,0.892217,0.553640,0.595407,6.564951,0.989439,16.210519,15.368212


Post-warmup front-tenor comparison:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
18,ewma_10_bias_adjusted,9,1742.0,-0.065011,0.734333,0.963242,0.585135,0.602348,6.105521,0.933536,15.966129,14.524413
9,ewma_10_raw,9,1742.0,0.252932,0.758145,0.989636,0.638577,0.604832,8.053205,1.270484,15.966129,16.986933
0,trailing_realized_variance_baseline,9,1742.0,0.084441,0.830584,1.098824,0.649170,0.546758,32.734106,1.128212,15.966129,16.512511
19,ewma_10_bias_adjusted,12,1741.0,-0.057069,0.703504,0.924414,0.586784,0.599223,6.236205,0.973706,15.772054,14.604556
10,ewma_10_raw,12,1741.0,0.211245,0.727495,0.940757,0.628195,0.602258,7.863730,1.260988,15.772054,16.643763
1,trailing_realized_variance_baseline,12,1741.0,0.053141,0.785446,1.040216,0.629956,0.550582,48.255231,1.159359,15.772054,16.129914
20,ewma_10_bias_adjusted,15,1738.0,-0.048542,0.684224,0.903909,0.572479,0.596647,6.435510,0.989614,16.361159,15.384996
11,ewma_10_raw,15,1738.0,0.173571,0.702894,0.912000,0.574457,0.600245,7.770278,1.236531,16.361159,17.123509
2,trailing_realized_variance_baseline,15,1738.0,0.031874,0.754674,1.007330,0.621250,0.544085,83.284308,1.147431,16.361159,16.530656
21,ewma_10_bias_adjusted,18,1736.0,-0.046050,0.674080,0.892217,0.553640,0.595407,6.564951,0.989439,16.210519,15.368212


In [19]:
# ============================================================
# Cell 14: Save bias-adjusted forecast panel and audit outputs
# ============================================================

bias_adjusted_df.to_parquet(
    VOL_FORECAST_EWMA_BIAS_ADJUSTED_PANEL_PARQUET,
    index=False,
)

bias_comparison_df.to_csv(
    VOL_FORECAST_EWMA_BIAS_COMPARISON_CSV,
    index=False,
)

bias_comparison_by_tenor_df.to_csv(
    VOL_FORECAST_EWMA_BIAS_COMPARISON_BY_TENOR_CSV,
    index=False,
)

post_warmup_comparison_df.to_csv(
    VOL_FORECAST_EWMA_BIAS_COMPARISON_POST_WARMUP_CSV,
    index=False,
)

post_warmup_comparison_by_tenor_df.to_csv(
    VOL_FORECAST_EWMA_BIAS_COMPARISON_BY_TENOR_POST_WARMUP_CSV,
    index=False,
)

print("Saved bias-adjusted EWMA panel:", VOL_FORECAST_EWMA_BIAS_ADJUSTED_PANEL_PARQUET)
print("Saved bias comparison:", VOL_FORECAST_EWMA_BIAS_COMPARISON_CSV)
print("Saved bias comparison by tenor:", VOL_FORECAST_EWMA_BIAS_COMPARISON_BY_TENOR_CSV)
print("Saved post-warmup comparison:", VOL_FORECAST_EWMA_BIAS_COMPARISON_POST_WARMUP_CSV)
print("Saved post-warmup comparison by tenor:", VOL_FORECAST_EWMA_BIAS_COMPARISON_BY_TENOR_POST_WARMUP_CSV)

Saved bias-adjusted EWMA panel: C:\Users\patri\vrp_project\data\processed\vol_forecast_ewma_bias_adjusted_panel_v0_1.parquet
Saved bias comparison: C:\Users\patri\vrp_project\data\audit\vol_forecast_ewma_bias_comparison_v0_1.csv
Saved bias comparison by tenor: C:\Users\patri\vrp_project\data\audit\vol_forecast_ewma_bias_comparison_by_tenor_v0_1.csv
Saved post-warmup comparison: C:\Users\patri\vrp_project\data\audit\vol_forecast_ewma_bias_comparison_post_warmup_v0_1.csv
Saved post-warmup comparison by tenor: C:\Users\patri\vrp_project\data\audit\vol_forecast_ewma_bias_comparison_by_tenor_post_warmup_v0_1.csv


In [20]:
# ============================================================
# Cell 15: Scheduled forward trading-day counts
# ============================================================

import pandas_market_calendars as mcal

SCHEDULED_COUNT_AUDIT_CSV = (
    AUDIT_DIR / "vol_forecast_scheduled_count_audit_v0_1.csv"
)

# Use NYSE calendar as proxy for SPX trading days.
nyse = mcal.get_calendar("XNYS")

min_schedule_date = pd.to_datetime(
    str(int(bias_adjusted_df["trade_date"].min())),
    format="%Y%m%d",
)

max_schedule_date = pd.to_datetime(
    str(int(bias_adjusted_df["trade_date"].max())),
    format="%Y%m%d",
) + pd.Timedelta(days=max(TENORS) + 10)

nyse_schedule = nyse.schedule(
    start_date=min_schedule_date.strftime("%Y-%m-%d"),
    end_date=max_schedule_date.strftime("%Y-%m-%d"),
)

scheduled_trade_dates = (
    pd.to_datetime(nyse_schedule.index)
    .normalize()
    .tz_localize(None)
)

scheduled_trade_dates_np = scheduled_trade_dates.values.astype("datetime64[ns]")

print("Scheduled NYSE trading dates:", len(scheduled_trade_dates))
print("Schedule range:", scheduled_trade_dates.min(), "to", scheduled_trade_dates.max())

# Build scheduled count for each trade_date x tenor.
scheduled_count_rows = []

unique_trade_dates = sorted(bias_adjusted_df["trade_date"].unique())

for trade_date in unique_trade_dates:
    trade_ts = pd.to_datetime(str(int(trade_date)), format="%Y%m%d")

    for tenor in TENORS:
        end_ts = trade_ts + pd.Timedelta(days=int(tenor))

        start_idx = np.searchsorted(
            scheduled_trade_dates_np,
            np.datetime64(trade_ts),
            side="right",
        )

        end_idx = np.searchsorted(
            scheduled_trade_dates_np,
            np.datetime64(end_ts),
            side="right",
        )

        scheduled_count_rows.append({
            "trade_date": int(trade_date),
            "tenor": int(tenor),
            "scheduled_forward_num_returns": int(end_idx - start_idx),
        })

scheduled_count_df = pd.DataFrame(scheduled_count_rows)

print("Scheduled count rows:", len(scheduled_count_df))

display(
    scheduled_count_df
    .groupby("tenor")["scheduled_forward_num_returns"]
    .describe()
)

# Compare scheduled counts to realized forward counts where realized target exists.
scheduled_count_check_df = (
    bias_adjusted_df[
        [
            "trade_date",
            "tenor",
            "forward_num_returns",
            "forward_realized_variance",
        ]
    ]
    .merge(
        scheduled_count_df,
        on=["trade_date", "tenor"],
        how="left",
        validate="one_to_one",
    )
)

scheduled_count_check_df["count_diff_scheduled_minus_realized"] = (
    scheduled_count_check_df["scheduled_forward_num_returns"]
    - scheduled_count_check_df["forward_num_returns"]
)

scheduled_count_audit_df = (
    scheduled_count_check_df
    .groupby("tenor")
    .agg(
        rows=("trade_date", "count"),
        rows_with_realized_forward_target=("forward_realized_variance", lambda x: x.notna().sum()),
        avg_scheduled_count=("scheduled_forward_num_returns", "mean"),
        median_scheduled_count=("scheduled_forward_num_returns", "median"),
        min_scheduled_count=("scheduled_forward_num_returns", "min"),
        max_scheduled_count=("scheduled_forward_num_returns", "max"),
        avg_realized_count=("forward_num_returns", "mean"),
        median_realized_count=("forward_num_returns", "median"),
        min_realized_count=("forward_num_returns", "min"),
        max_realized_count=("forward_num_returns", "max"),
        avg_count_diff=("count_diff_scheduled_minus_realized", "mean"),
        max_abs_count_diff=("count_diff_scheduled_minus_realized", lambda x: x.abs().max()),
    )
    .reset_index()
)

display(scheduled_count_audit_df)

scheduled_count_audit_df.to_csv(
    SCHEDULED_COUNT_AUDIT_CSV,
    index=False,
)

print("Saved scheduled count audit:", SCHEDULED_COUNT_AUDIT_CSV)

print("\nLatest date scheduled counts:")
display(
    scheduled_count_df[
        scheduled_count_df["trade_date"] == scheduled_count_df["trade_date"].max()
    ]
)

Scheduled NYSE trading dates: 2041
Schedule range: 2018-06-25 00:00:00 to 2026-08-07 00:00:00
Scheduled count rows: 18099


,count,mean,std,min,25%,50%,75%,max
tenor,,,,,,,,
9,2011.0,6.171059,0.893079,4.0,5.5,6.0,7.0,7.0
12,2011.0,7.873695,0.630059,6.0,7.0,8.0,8.0,9.0
15,2011.0,10.400796,0.665780,8.0,10.0,10.0,11.0,11.0
18,2011.0,12.103928,0.972067,10.0,11.0,12.0,13.0,14.0
21,2011.0,14.431129,0.618621,12.0,14.0,14.0,15.0,15.0
24,2011.0,16.355545,1.094715,13.0,16.0,16.0,17.0,18.0
27,2011.0,18.281949,0.676226,15.0,18.0,18.0,19.0,19.0
30,2011.0,20.609150,1.055851,17.0,20.0,21.0,21.0,22.0
33,2011.0,22.312780,0.838223,19.0,22.0,22.0,23.0,24.0


,tenor,rows,rows_with_realized_forward_target,avg_scheduled_count,median_scheduled_count,min_scheduled_count,max_scheduled_count,avg_realized_count,median_realized_count,min_realized_count,max_realized_count,avg_count_diff,max_abs_count_diff
0,9,2011,2010,6.171059,6.0,4,7,6.160617,6.0,0,7,0.010443,5
1,12,2011,2010,7.873695,8.0,6,9,7.858777,8.0,0,9,0.014918,7
2,15,2011,2010,10.400796,10.0,8,11,10.374441,10.0,0,11,0.026355,10
3,18,2011,2010,12.103928,12.0,10,14,12.069120,12.0,0,14,0.034809,11
4,21,2011,2010,14.431129,14.0,12,15,14.381402,14.0,0,15,0.049727,14
5,24,2011,2010,16.355545,16.0,13,18,16.291397,16.0,0,18,0.064147,15
6,27,2011,2010,18.281949,18.0,15,19,18.201392,18.0,0,19,0.080557,18
7,30,2011,2010,20.609150,21.0,17,22,20.505719,21.0,0,22,0.103431,20
8,33,2011,2010,22.312780,22.0,19,24,22.191944,22.0,0,24,0.120835,22


Saved scheduled count audit: C:\Users\patri\vrp_project\data\audit\vol_forecast_scheduled_count_audit_v0_1.csv

Latest date scheduled counts:


,trade_date,tenor,scheduled_forward_num_returns
18090,20260625,9,5
18091,20260625,12,7
18092,20260625,15,10
18093,20260625,18,11
18094,20260625,21,14
18095,20260625,24,15
18096,20260625,27,18
18097,20260625,30,20
18098,20260625,33,22


In [21]:
# ============================================================
# Cell 16: Rebuild EWMA 10 forecasts using scheduled counts
# ============================================================

simple_forecast_df = bias_adjusted_df.copy()

# Remove previous scheduled count if rerunning this cell.
if "scheduled_forward_num_returns" in simple_forecast_df.columns:
    simple_forecast_df = simple_forecast_df.drop(columns=["scheduled_forward_num_returns"])

simple_forecast_df = simple_forecast_df.merge(
    scheduled_count_df,
    on=["trade_date", "tenor"],
    how="left",
    validate="one_to_one",
)

if simple_forecast_df["scheduled_forward_num_returns"].isna().any():
    raise ValueError("Some rows are missing scheduled_forward_num_returns.")

scheduled_count = simple_forecast_df["scheduled_forward_num_returns"].replace(0, np.nan)
tenor_year_fraction = simple_forecast_df["tenor"] / 365.0

# ----------------------------
# Raw scheduled EWMA 10 forecast
# ----------------------------

simple_forecast_df["ewma_10_scheduled_forecast_variance"] = (
    simple_forecast_df["ewma_10_daily_variance"]
    * scheduled_count
    / tenor_year_fraction
)

simple_forecast_df["ewma_10_scheduled_forecast_vol"] = (
    np.sqrt(simple_forecast_df["ewma_10_scheduled_forecast_variance"])
    * 100.0
)

simple_forecast_df["log_ewma_10_scheduled_forecast_variance"] = np.log(
    simple_forecast_df["ewma_10_scheduled_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

simple_forecast_df["ewma_10_scheduled_vrp_signal"] = np.log(
    simple_forecast_df["implied_variance"].clip(lower=EPSILON_VARIANCE)
    / simple_forecast_df["ewma_10_scheduled_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

simple_forecast_df["ewma_10_scheduled_vrp_vol_ratio"] = (
    simple_forecast_df["vix_style_vol"]
    / simple_forecast_df["ewma_10_scheduled_forecast_vol"]
)

# ----------------------------
# Bias-adjusted scheduled EWMA 10 forecast
# ----------------------------

simple_forecast_df["log_ewma_10_scheduled_bias_adjusted_forecast_variance_strict"] = (
    simple_forecast_df["log_ewma_10_scheduled_forecast_variance"]
    + simple_forecast_df["ewma_10_bias_adjustment_log"]
)

simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_variance_strict"] = np.exp(
    simple_forecast_df["log_ewma_10_scheduled_bias_adjusted_forecast_variance_strict"]
)

simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_vol_strict"] = (
    np.sqrt(simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_variance_strict"])
    * 100.0
)

# Fallback: use bias-adjusted where available, otherwise raw scheduled EWMA 10.
simple_forecast_df["log_ewma_10_scheduled_bias_adjusted_forecast_variance"] = (
    simple_forecast_df["log_ewma_10_scheduled_bias_adjusted_forecast_variance_strict"]
    .fillna(simple_forecast_df["log_ewma_10_scheduled_forecast_variance"])
)

simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_variance"] = np.exp(
    simple_forecast_df["log_ewma_10_scheduled_bias_adjusted_forecast_variance"]
)

simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_vol"] = (
    np.sqrt(simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_variance"])
    * 100.0
)

simple_forecast_df["ewma_10_scheduled_bias_adjusted_vrp_signal"] = np.log(
    simple_forecast_df["implied_variance"].clip(lower=EPSILON_VARIANCE)
    / simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

simple_forecast_df["ewma_10_scheduled_bias_adjusted_vrp_vol_ratio"] = (
    simple_forecast_df["vix_style_vol"]
    / simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_vol"]
)

# ----------------------------
# Hybrid simple forecast rule
# ----------------------------
# Based on prior results:
#   9-18d: bias-adjusted EWMA 10 helps.
#   21-33d: raw EWMA 10 is slightly better.
#
# If bias adjustment is unavailable in the warmup period, fallback to raw EWMA 10.

simple_forecast_df["simple_forecast_model"] = np.where(
    (simple_forecast_df["tenor"] <= 18)
    & simple_forecast_df["ewma_10_bias_adjustment_log"].notna(),
    "ewma_10_scheduled_bias_adjusted",
    "ewma_10_scheduled_raw",
)

simple_forecast_df["simple_forecast_variance"] = np.where(
    simple_forecast_df["simple_forecast_model"] == "ewma_10_scheduled_bias_adjusted",
    simple_forecast_df["ewma_10_scheduled_bias_adjusted_forecast_variance"],
    simple_forecast_df["ewma_10_scheduled_forecast_variance"],
)

simple_forecast_df["simple_forecast_vol"] = (
    np.sqrt(simple_forecast_df["simple_forecast_variance"])
    * 100.0
)

simple_forecast_df["log_simple_forecast_variance"] = np.log(
    simple_forecast_df["simple_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

simple_forecast_df["simple_forecast_vrp_signal"] = np.log(
    simple_forecast_df["implied_variance"].clip(lower=EPSILON_VARIANCE)
    / simple_forecast_df["simple_forecast_variance"].clip(lower=EPSILON_VARIANCE)
)

simple_forecast_df["simple_forecast_vrp_vol_ratio"] = (
    simple_forecast_df["vix_style_vol"]
    / simple_forecast_df["simple_forecast_vol"]
)

print("Simple forecast panel rows:", len(simple_forecast_df))

print("\nMissing scheduled forecast fields:")
display(
    simple_forecast_df[
        [
            "scheduled_forward_num_returns",
            "ewma_10_scheduled_forecast_variance",
            "ewma_10_scheduled_bias_adjusted_forecast_variance",
            "simple_forecast_variance",
            "simple_forecast_vrp_signal",
        ]
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

print("\nSimple forecast model counts:")
display(
    simple_forecast_df["simple_forecast_model"]
    .value_counts()
    .rename("rows")
    .to_frame()
)

print("\nLatest date simple forecast rows:")
display(
    simple_forecast_df[
        simple_forecast_df["trade_date"] == simple_forecast_df["trade_date"].max()
    ][
        [
            "trade_date",
            "tenor",
            "scheduled_forward_num_returns",
            "vix_style_vol",
            "ewma_10_scheduled_forecast_vol",
            "ewma_10_bias_adjustment_log",
            "ewma_10_scheduled_bias_adjusted_forecast_vol",
            "simple_forecast_model",
            "simple_forecast_vol",
            "simple_forecast_vrp_signal",
            "forward_realized_vol",
        ]
    ]
)

Simple forecast panel rows: 18099

Missing scheduled forecast fields:


,missing_count
scheduled_forward_num_returns,0
ewma_10_scheduled_forecast_variance,81
ewma_10_scheduled_bias_adjusted_forecast_variance,81
simple_forecast_variance,81
simple_forecast_vrp_signal,81



Simple forecast model counts:


,rows
simple_forecast_model,
ewma_10_scheduled_raw,11138
ewma_10_scheduled_bias_adjusted,6961



Latest date simple forecast rows:


,trade_date,tenor,scheduled_forward_num_returns,vix_style_vol,ewma_10_scheduled_forecast_vol,ewma_10_bias_adjustment_log,ewma_10_scheduled_bias_adjusted_forecast_vol,simple_forecast_model,simple_forecast_vol,simple_forecast_vrp_signal,forward_realized_vol
18090,20260625,9,5,17.952376,13.919719,-0.264548,12.195079,ewma_10_scheduled_bias_adjusted,12.195079,0.773380,NaN
18091,20260625,12,7,17.964763,14.263468,-0.222528,12.761558,ewma_10_scheduled_bias_adjusted,12.761558,0.683950,NaN
18092,20260625,15,10,17.972192,15.248289,-0.180657,13.931312,ewma_10_scheduled_bias_adjusted,13.931312,0.509373,NaN
18093,20260625,18,11,17.707876,14.599125,-0.153976,13.517342,ewma_10_scheduled_bias_adjusted,13.517342,0.540072,NaN
18094,20260625,21,14,17.516638,15.248289,-0.135155,14.251894,ewma_10_scheduled_raw,15.248289,0.277368,NaN
18095,20260625,24,15,18.028765,14.764092,-0.112047,13.959702,ewma_10_scheduled_raw,14.764092,0.399541,NaN
18096,20260625,27,18,18.596577,15.248289,-0.101351,14.494825,ewma_10_scheduled_raw,15.248289,0.397020,NaN
18097,20260625,30,20,18.980878,15.248289,-0.082188,14.634378,ewma_10_scheduled_raw,15.248289,0.437930,NaN
18098,20260625,33,22,19.185842,15.248289,-0.073028,14.701557,ewma_10_scheduled_raw,15.248289,0.459411,NaN


In [22]:
# ============================================================
# Cell 17: Evaluate scheduled-count forecasts
# ============================================================

def evaluate_variance_forecast_v2(
    df,
    model_name,
    forecast_variance_col,
    group_cols=None,
):
    """
    Evaluate a variance forecast versus forward realized variance.
    Uses log-variance error metrics.
    """
    if group_cols is None:
        group_cols = []

    temp = df[
        df[forecast_variance_col].notna()
        & df["forward_realized_variance"].notna()
        & (df[forecast_variance_col] > 0)
        & (df["forward_realized_variance"] > 0)
    ].copy()

    temp["log_forecast_variance_eval"] = np.log(
        temp[forecast_variance_col].clip(lower=EPSILON_VARIANCE)
    )

    temp["log_forward_realized_variance_eval"] = np.log(
        temp["forward_realized_variance"].clip(lower=EPSILON_VARIANCE)
    )

    temp["log_error"] = (
        temp["log_forecast_variance_eval"]
        - temp["log_forward_realized_variance_eval"]
    )

    temp["abs_log_error"] = temp["log_error"].abs()
    temp["squared_log_error"] = temp["log_error"] ** 2

    temp["forecast_to_realized_variance_ratio"] = (
        temp[forecast_variance_col]
        / temp["forward_realized_variance"]
    )

    def summarize_group(x):
        return {
            "rows": len(x),
            "bias_log_forecast_minus_realized": x["log_error"].mean(),
            "mae_log": x["abs_log_error"].mean(),
            "rmse_log": np.sqrt(x["squared_log_error"].mean()),
            "median_abs_log_error": x["abs_log_error"].median(),
            "corr_log_forecast_vs_realized": x["log_forecast_variance_eval"].corr(
                x["log_forward_realized_variance_eval"]
            ),
            "avg_forecast_to_realized_variance_ratio": (
                x["forecast_to_realized_variance_ratio"].mean()
            ),
            "median_forecast_to_realized_variance_ratio": (
                x["forecast_to_realized_variance_ratio"].median()
            ),
            "avg_forward_realized_vol": x["forward_realized_vol"].mean(),
            "avg_forecast_vol": (np.sqrt(x[forecast_variance_col]) * 100.0).mean(),
        }

    if not group_cols:
        out = pd.DataFrame([summarize_group(temp)])
    else:
        rows = []

        for key, group in temp.groupby(group_cols, dropna=False):
            if not isinstance(key, tuple):
                key = (key,)

            row = dict(zip(group_cols, key))
            row.update(summarize_group(group))
            rows.append(row)

        out = pd.DataFrame(rows)

    out.insert(0, "model", model_name)

    return out


scheduled_model_specs = [
    {
        "model": "trailing_realized_variance_baseline",
        "forecast_variance_col": "trailing_realized_variance",
    },
    {
        "model": "ewma_10_scheduled_raw",
        "forecast_variance_col": "ewma_10_scheduled_forecast_variance",
    },
    {
        "model": "ewma_10_scheduled_bias_adjusted",
        "forecast_variance_col": "ewma_10_scheduled_bias_adjusted_forecast_variance",
    },
    {
        "model": "simple_hybrid_front_adjusted_back_raw",
        "forecast_variance_col": "simple_forecast_variance",
    },
]

scheduled_eval_rows = []
scheduled_tenor_eval_rows = []

for spec in scheduled_model_specs:
    scheduled_eval_rows.append(
        evaluate_variance_forecast_v2(
            df=simple_forecast_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=[],
        )
    )

    scheduled_tenor_eval_rows.append(
        evaluate_variance_forecast_v2(
            df=simple_forecast_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=["tenor"],
        )
    )

scheduled_comparison_df = pd.concat(
    scheduled_eval_rows,
    ignore_index=True,
)

scheduled_comparison_by_tenor_df = pd.concat(
    scheduled_tenor_eval_rows,
    ignore_index=True,
)

print("Scheduled-count overall comparison:")
display(
    scheduled_comparison_df.sort_values("rmse_log")
)

print("Scheduled-count comparison by tenor:")
display(
    scheduled_comparison_by_tenor_df.sort_values(["tenor", "rmse_log"])
)

print("Scheduled-count front-tenor comparison:")
display(
    scheduled_comparison_by_tenor_df[
        scheduled_comparison_by_tenor_df["tenor"].isin([9, 12, 15, 18])
    ].sort_values(["tenor", "rmse_log"])
)

print("Scheduled-count back-tenor comparison:")
display(
    scheduled_comparison_by_tenor_df[
        scheduled_comparison_by_tenor_df["tenor"].isin([21, 24, 27, 30, 33])
    ].sort_values(["tenor", "rmse_log"])
)

# Same-sample post-warmup comparison.
scheduled_post_warmup_df = simple_forecast_df[
    simple_forecast_df["ewma_10_bias_adjustment_log"].notna()
].copy()

scheduled_post_warmup_eval_rows = []
scheduled_post_warmup_tenor_eval_rows = []

for spec in scheduled_model_specs:
    scheduled_post_warmup_eval_rows.append(
        evaluate_variance_forecast_v2(
            df=scheduled_post_warmup_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=[],
        )
    )

    scheduled_post_warmup_tenor_eval_rows.append(
        evaluate_variance_forecast_v2(
            df=scheduled_post_warmup_df,
            model_name=spec["model"],
            forecast_variance_col=spec["forecast_variance_col"],
            group_cols=["tenor"],
        )
    )

scheduled_post_warmup_comparison_df = pd.concat(
    scheduled_post_warmup_eval_rows,
    ignore_index=True,
)

scheduled_post_warmup_comparison_by_tenor_df = pd.concat(
    scheduled_post_warmup_tenor_eval_rows,
    ignore_index=True,
)

print("Scheduled-count post-warmup overall comparison:")
display(
    scheduled_post_warmup_comparison_df.sort_values("rmse_log")
)

print("Scheduled-count post-warmup by tenor:")
display(
    scheduled_post_warmup_comparison_by_tenor_df.sort_values(["tenor", "rmse_log"])
)

Scheduled-count overall comparison:


,model,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
3,simple_hybrid_front_adjusted_back_raw,18009,0.061836,0.705209,0.946438,0.562768,0.540285,70.998258,1.088437,15.933831,15.857664
2,ewma_10_scheduled_bias_adjusted,18009,0.002344,0.705808,0.948695,0.563069,0.536833,65.997572,1.025070,15.933831,15.416886
1,ewma_10_scheduled_raw,18009,0.158695,0.712549,0.951918,0.568117,0.542761,74.663104,1.203116,15.933831,16.605138
0,trailing_realized_variance_baseline,18090,0.033139,0.744142,0.999440,0.595182,0.534185,87.655811,1.096107,15.902944,16.091999


Scheduled-count comparison by tenor:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
18,ewma_10_scheduled_bias_adjusted,9,2001,-0.002676,0.752924,0.992071,0.606560,0.572682,25.780873,0.997657,15.556921,14.506369
27,simple_hybrid_front_adjusted_back_raw,9,2001,-0.002676,0.752924,0.992071,0.606560,0.572682,25.780873,0.997657,15.556921,14.506369
9,ewma_10_scheduled_raw,9,2001,0.274113,0.773654,1.014995,0.660012,0.586021,33.585124,1.308569,15.556921,16.651789
0,trailing_realized_variance_baseline,9,2010,0.083543,0.826110,1.091774,0.643395,0.556045,28.601153,1.134990,15.530672,16.063016
19,ewma_10_scheduled_bias_adjusted,12,2001,-0.000612,0.722249,0.953195,0.605325,0.568581,30.889237,1.042215,15.364099,14.540848
28,simple_hybrid_front_adjusted_back_raw,12,2001,-0.000612,0.722249,0.953195,0.605325,0.568581,30.889237,1.042215,15.364099,14.540848
10,ewma_10_scheduled_raw,12,2001,0.232838,0.743123,0.967626,0.637244,0.580780,38.590119,1.300010,15.364099,16.316710
1,trailing_realized_variance_baseline,12,2010,0.054021,0.777463,1.029848,0.627210,0.563261,42.003096,1.163674,15.338475,15.696962
20,ewma_10_scheduled_bias_adjusted,15,2001,-0.000936,0.709035,0.939814,0.591947,0.557454,45.265928,1.037806,15.952008,15.266102
29,simple_hybrid_front_adjusted_back_raw,15,2001,-0.000936,0.709035,0.939814,0.591947,0.557454,45.265928,1.037806,15.952008,15.266102


Scheduled-count front-tenor comparison:


...

,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
22,ewma_10_scheduled_bias_adjusted,21,2001,0.004759,0.700533,0.937046,0.569901,0.534486,65.634047,1.030380,16.060596,15.613639
13,ewma_10_scheduled_raw,21,2001,0.147141,0.705719,0.937722,0.555463,0.541872,75.175068,1.207086,16.060596,16.705057
31,simple_hybrid_front_adjusted_back_raw,21,2001,0.147141,0.705719,0.937722,0.555463,0.541872,75.175068,1.207086,16.060596,16.705057
4,trailing_realized_variance_baseline,21,2010,0.009755,0.729746,0.977445,0.582302,0.537138,110.465463,1.091401,16.028385,16.036072
14,ewma_10_scheduled_raw,24,2001,0.123330,0.692373,0.934114,0.535810,0.532113,85.703383,1.144525,16.095895,16.628914
32,simple_hybrid_front_adjusted_back_raw,24,2001,0.123330,0.692373,0.934114,0.535810,0.532113,85.703383,1.144525,16.095895,16.628914
23,ewma_10_scheduled_bias_adjusted,24,2001,0.001317,0.692933,0.936412,0.539385,0.525576,76.622138,1.017456,16.095895,15.704913
5,trailing_realized_variance_baseline,24,2010,0.026423,0.726626,0.975950,0.585444,0.523549,113.695428,1.066770,16.062740,16.219301
15,ewma_10_scheduled_raw,27,2001,0.113153,0.692209,0.938443,0.545871,0.517351,96.143360,1.135163,16.084744,16.581209
33,simple_hybrid_front_adjusted_back_raw,27,2001,0.113153,0.692209,0.938443,0.545871,0.517351,96.143360,1.135163,16.084744,16.581209


Scheduled-count post-warmup overall comparison:


,model,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
3,simple_hybrid_front_adjusted_back_raw,15605,0.040279,0.678260,0.925688,0.534746,0.563200,81.645963,1.055715,16.30343,16.122644
2,ewma_10_scheduled_bias_adjusted,15605,-0.028377,0.678952,0.928349,0.534258,0.561564,75.874905,0.985478,16.30343,15.613963
1,ewma_10_scheduled_raw,15605,0.152060,0.686731,0.932150,0.539588,0.563342,85.875390,1.180965,16.30343,16.985269
0,trailing_realized_variance_baseline,15605,0.044294,0.734421,0.990456,0.590531,0.538563,101.375835,1.092695,16.30343,16.568702


Scheduled-count post-warmup by tenor:


,model,tenor,rows,bias_log_forecast_minus_realized,mae_log,rmse_log,median_abs_log_error,corr_log_forecast_vs_realized,avg_forecast_to_realized_variance_ratio,median_forecast_to_realized_variance_ratio,avg_forward_realized_vol,avg_forecast_vol
18,ewma_10_scheduled_bias_adjusted,9,1742,-0.062544,0.736800,0.978358,0.586871,0.588203,29.274469,0.933536,15.966129,14.537705
27,simple_hybrid_front_adjusted_back_raw,9,1742,-0.062544,0.736800,0.978358,0.586871,0.588203,29.274469,0.933536,15.966129,14.537705
9,ewma_10_scheduled_raw,9,1742,0.255399,0.760612,1.005005,0.639637,0.590670,38.239054,1.270484,15.966129,17.002106
0,trailing_realized_variance_baseline,9,1742,0.084441,0.830584,1.098824,0.649170,0.546758,32.734106,1.128212,15.966129,16.512511
19,ewma_10_scheduled_bias_adjusted,12,1741,-0.053988,0.706584,0.941634,0.587622,0.582552,35.193091,0.973706,15.772054,14.620307
28,simple_hybrid_front_adjusted_back_raw,12,1741,-0.053988,0.706584,0.941634,0.587622,0.582552,35.193091,0.973706,15.772054,14.620307
10,ewma_10_scheduled_raw,12,1741,0.214326,0.730576,0.958400,0.630314,0.585504,44.044019,1.260988,15.772054,16.661375
1,trailing_realized_variance_baseline,12,1741,0.053141,0.785446,1.040216,0.629956,0.550582,48.255231,1.159359,15.772054,16.129914
20,ewma_10_scheduled_bias_adjusted,15,1738,-0.044105,0.688476,0.926330,0.574795,0.574439,51.824814,0.990277,16.361159,15.408540
29,simple_hybrid_front_adjusted_back_raw,15,1738,-0.044105,0.688476,0.926330,0.574795,0.574439,51.824814,0.990277,16.361159,15.408540


In [23]:
# ============================================================
# Cell 18: Save final simple forecast panel and audits
# ============================================================

VOL_FORECAST_SIMPLE_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "vol_forecast_simple_panel_v0_1.parquet"
)

FORECAST_VRP_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vrp_panel_v0_1.parquet"
)

VOL_FORECAST_SIMPLE_COMPARISON_CSV = (
    AUDIT_DIR / "vol_forecast_simple_comparison_v0_1.csv"
)

VOL_FORECAST_SIMPLE_COMPARISON_BY_TENOR_CSV = (
    AUDIT_DIR / "vol_forecast_simple_comparison_by_tenor_v0_1.csv"
)

VOL_FORECAST_SIMPLE_POST_WARMUP_COMPARISON_CSV = (
    AUDIT_DIR / "vol_forecast_simple_post_warmup_comparison_v0_1.csv"
)

VOL_FORECAST_SIMPLE_POST_WARMUP_COMPARISON_BY_TENOR_CSV = (
    AUDIT_DIR / "vol_forecast_simple_post_warmup_comparison_by_tenor_v0_1.csv"
)

# Full panel for research.
simple_forecast_df.to_parquet(
    VOL_FORECAST_SIMPLE_PANEL_PARQUET,
    index=False,
)

# Smaller panel for next notebook's signal work.
forecast_vrp_cols = [
    "trade_date",
    "tenor",
    "spx_close",
    "spx_rsi_14",
    "vix_style_vol",
    "implied_variance",
    "trailing_realized_variance",
    "trailing_realized_vol",
    "primary_vrp_signal",
    "scheduled_forward_num_returns",
    "ewma_10_scheduled_forecast_variance",
    "ewma_10_scheduled_forecast_vol",
    "ewma_10_scheduled_vrp_signal",
    "ewma_10_bias_adjustment_log",
    "ewma_10_scheduled_bias_adjusted_forecast_variance",
    "ewma_10_scheduled_bias_adjusted_forecast_vol",
    "ewma_10_scheduled_bias_adjusted_vrp_signal",
    "simple_forecast_model",
    "simple_forecast_variance",
    "simple_forecast_vol",
    "simple_forecast_vrp_signal",
    "simple_forecast_vrp_vol_ratio",
    "forward_realized_variance",
    "forward_realized_vol",
]

available_forecast_vrp_cols = [
    c for c in forecast_vrp_cols
    if c in simple_forecast_df.columns
]

forecast_vrp_panel_df = simple_forecast_df[
    available_forecast_vrp_cols
].copy()

forecast_vrp_panel_df.to_parquet(
    FORECAST_VRP_PANEL_PARQUET,
    index=False,
)

scheduled_comparison_df.to_csv(
    VOL_FORECAST_SIMPLE_COMPARISON_CSV,
    index=False,
)

scheduled_comparison_by_tenor_df.to_csv(
    VOL_FORECAST_SIMPLE_COMPARISON_BY_TENOR_CSV,
    index=False,
)

scheduled_post_warmup_comparison_df.to_csv(
    VOL_FORECAST_SIMPLE_POST_WARMUP_COMPARISON_CSV,
    index=False,
)

scheduled_post_warmup_comparison_by_tenor_df.to_csv(
    VOL_FORECAST_SIMPLE_POST_WARMUP_COMPARISON_BY_TENOR_CSV,
    index=False,
)

print("Saved simple forecast panel:", VOL_FORECAST_SIMPLE_PANEL_PARQUET)
print("Saved forecast VRP panel:", FORECAST_VRP_PANEL_PARQUET)
print("Saved simple comparison:", VOL_FORECAST_SIMPLE_COMPARISON_CSV)
print("Saved simple comparison by tenor:", VOL_FORECAST_SIMPLE_COMPARISON_BY_TENOR_CSV)
print("Saved post-warmup comparison:", VOL_FORECAST_SIMPLE_POST_WARMUP_COMPARISON_CSV)
print("Saved post-warmup comparison by tenor:", VOL_FORECAST_SIMPLE_POST_WARMUP_COMPARISON_BY_TENOR_CSV)

print("\nForecast VRP panel rows:", len(forecast_vrp_panel_df))
print("Forecast VRP panel date range:", forecast_vrp_panel_df["trade_date"].min(), "to", forecast_vrp_panel_df["trade_date"].max())
print("Forecast VRP panel tenors:", sorted(forecast_vrp_panel_df["tenor"].unique()))

Saved simple forecast panel: C:\Users\patri\vrp_project\data\processed\vol_forecast_simple_panel_v0_1.parquet
Saved forecast VRP panel: C:\Users\patri\vrp_project\data\processed\forecast_vrp_panel_v0_1.parquet
Saved simple comparison: C:\Users\patri\vrp_project\data\audit\vol_forecast_simple_comparison_v0_1.csv
Saved simple comparison by tenor: C:\Users\patri\vrp_project\data\audit\vol_forecast_simple_comparison_by_tenor_v0_1.csv
Saved post-warmup comparison: C:\Users\patri\vrp_project\data\audit\vol_forecast_simple_post_warmup_comparison_v0_1.csv
Saved post-warmup comparison by tenor: C:\Users\patri\vrp_project\data\audit\vol_forecast_simple_post_warmup_comparison_by_tenor_v0_1.csv

Forecast VRP panel rows: 18099
Forecast VRP panel date range: 20180625 to 20260625
Forecast VRP panel tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]
